In [35]:
# --- Program Title Bucketing (Crawl + CollegeVine) ---
# Notebook-friendly: define inputs/outputs here, run top-to-bottom.

import re
import html
import unicodedata
import pandas as pd
from pathlib import Path

# =========================
# Inputs / Outputs (EDIT ME)
# =========================
INPUT_CSV  = "ace_unitid_merge__ace_x_2013comp__webscrape__v15simple.csv"

OUTPUT_CSV = "ace_unitid_merge__ace_x_2013comp__webscrape__v15simple__bucketed_programs.csv"
OUTPUT_LONG_CSV = "ace_unitid_merge__ace_x_2013comp__webscrape__v15simple__bucketed_programs__long.csv"
OUTPUT_LONG_BUCKET_CSV = "ace_unitid_merge__ace_x_2013comp__webscrape__v15simple__bucketed_programs__long_bucket_summary.csv"

# NEW: clean program inventory + signals-only + aggregated program confidence
OUTPUT_LONG_PROGRAMS_CSV = "ace_unitid_merge__ace_x_2013comp__webscrape__v15simple__bucketed_programs__long_programs.csv"
OUTPUT_LONG_PROGRAMS_AGG_CSV = "ace_unitid_merge__ace_x_2013comp__webscrape__v15simple__bucketed_programs__long_programs_agg.csv"
OUTPUT_LONG_SIGNALS_CSV = "ace_unitid_merge__ace_x_2013comp__webscrape__v15simple__bucketed_programs__long_signals.csv"

# Column names in your file (adjust if needed)
CRAWL_TITLES_COL = "program_titles_found"
CV_TITLES_COL    = "college_vine_program_titles_found"

# Optional: reference column that often contains mojibake (edit if your file uses a different name)
REF_PROGRAM_NAME_COL = "2013_program_name"

# =========================
# Bucket Definitions
# =========================
# Precedence matters: first match wins (one title -> one bucket).
BUCKET_ORDER = [
    "black",      # African American / Black Studies (highest priority)
    "africana",
    "mena",       # Middle East / North Africa / Near East / Central Asia
    "african",    # generic African Studies (NOT African American)
    "minority",
    "ethnic",
    "race",
    "other",
]

# Real but non-program signals we want to track (centers/committees etc.)
REAL_NONPROGRAM_KINDS = ["center_institute", "committee_admin", "department_unit", "program_signal"]

# MENA trigger terms for exclusion and detection
MENA_TERMS = [
    "middle east", "middle eastern", "north africa", "north african",
    "mena", "near east", "near eastern", "central asian", "central asia",
]

# Regex patterns (precedence is controlled by BUCKET_ORDER)
BUCKET_PATTERNS = {
    "black": [
        r"\bafrican[-\s]?american\b",
        r"\bafro[-\s]?american\b",
        r"\bblack\s+studies\b",
        r"\bblack\s+(american|diaspora|diasporic)\b",
        r"\bafrican\s*,?\s*black\b",  # broad signal (keep)
        r"\bafrican\s*,?\s*black\s+and\s+caribbean\s+studies\b(?:\s+(?:program|department|major|minor|certificate|center|centre))?",  # e.g., "African, Black and Caribbean Studies"
        r"\bblack\s+and\s+.*\bstudies\b",  # e.g., "Black and Latino Studies"
        r"\bcritical\s+black\s+studies\b",
        r"\bafrican\s+and\s+african[-\s]+american\s+studies\b",  # robust to hyphen/space
        r"\bafrican\s+and\s+african\s+american\s+studies\b",  # legacy exact form
        r"\bblack\s+visual\s+cultures\b",
    ],
    "africana": [
        r"\bafricana\b",
        r"\bpan[-\s]?african\b",
        r"\bafricolog\w*\b",
    ],
    "mena": [
        r"\bmiddle\s+east\b",
        r"\bmiddle\s+eastern\b",
        r"\bnorth\s+africa\b",
        r"\bnorth\s+african\b",
        r"\bnear\s+east\b",
        r"\bnear\s+eastern\b",
        r"\bmena\b",
        r"\bmiddle\s+eastern\s+and\s+north\s+african\s+studies\b",
        r"\bmiddle\s+eastern\s+and\s+north\s+africa\s+studies\b",
        r"\bmiddle\s+east\s+and\s+north\s+africa\s+studies\b",
        r"\bmiddle\s+east\s+and\s+north\s+african\s+studies\b",
        r"\bmiddle\s+eastern\s*,\s*central\s+asian\s*,\s*and\s+north\s+african\s+studies\b",
    ],
    "african": [ #we exclude these from "African-American" bucket later
        r"\bafrican\s+studies\b",
        r"\bafrican\s+and\s+african\s+diaspora\s+studies\b",
        r"\bafrican\s+and\s+african\s+diasporic\s+studies\b",
        r"\bafrican\s+diaspora\s+studies\b",
        r"\bafrican\s+languages?\b",
        r"\bafrican\s+literatures?\b",
        r"\bafrican\s+language\s+program\b",
        r"\bafrica\b", 
    ],
    "minority": [
        r"\bminority\b",
        r"\bcultural\s+minority\b",
    ],
    "race": [
        r"\brace\b",
        r"\bracial\b",
        r"\bcritical\s+race\b",
        r"\bracial\s+justice\b",
        r"\brace\s*,?\s*power\b",
    ],
    "ethnic": [
        r"\brace\s*,?\s*ethnicity\b",
        r"\brace\s+and\s+ethnicity\b",
        r"\bethnic\b",
        r"\bethnicity\b",
    ],
}

# Strong indicators a string is an academic program title (vs. navigation/news/admin text)
# NOTE: intentionally does NOT include the word "studies" because it's too broad.
PROGRAM_TOKENS = [
    "major", "minor", "department", "program", "concentration",
    "certificate",
    "b.a", "ba", "b.s", "bs", "m.a", "ma", "m.s", "ms", "phd", "doctorate", "degree",
]

# Phrases that commonly indicate navigation/page chrome/admin/news (not a program title)
JUNK_PHRASES = [
    "welcome", "about", "view", "check out", "read more", "learn more", "connect with", "visit",
    "calendar", "faculty", "staff", "donate", "newsletter", "timeline",
    "annual report", "events", "news", "image", "photo", "video",
    "lecture series", "launchpad", "symposium", "workshop",
    "financial aid", "scholarship", "fellowship", "career map",
]
# Descriptor/support pages that are not themselves programs
DESCRIPTOR_NONPROGRAM_RE = re.compile(r"\b(financial\s+aid|scholarship|fellowship|career\s+map)\b", re.IGNORECASE)

# Common black-related false positives
BLACK_FALSE_POSITIVES = [
    "black belt", "blackstone", "blackboard", "black history month", "black student union",
]

# Heuristic verbs that often indicate trips/stories rather than programs
TRIP_STORY_VERBS = [
    "experience", "travel", "study abroad", "healing", "healthcare", "reads into", "honoring",
]

# Countries/regions commonly appearing in non-program strings (study abroad, course topics, stories)
COUNTRY_TERMS = [
    "ghana", "south africa", "nigeria", "kenya", "ethiopia", "uganda", "tanzania", "rwanda",
]

YEAR_PATTERN = re.compile(r"\b(19|20)\d{2}([\-/]\d{2})?\b")
CLASS_YEAR_PATTERN = re.compile(r"\b['\u2019]\d{2}\b")

# Course-like detection (run BEFORE bucket/program inference)
COURSE_PREFIX_RE = re.compile(
    r"^(introduction|intro|seminar|topics|special\s+topics|survey|advanced|capstone|independent\s+study|practicum|critical\s+debates|research|research\s+in)\b",
    flags=re.IGNORECASE,
)
COURSE_PHRASES = [
    "introduction to", "intro to", "critical debates", "seminar", "special topics", "research in",
    "4-year plan", "transfer plan", "curriculum", "requirements", "electives",
    "prerequisite", "prerequisites", "prereq",
]

# Major/minor shorthand evidence like "Africana (M, m)"
MM_SHORTHAND_RE = re.compile(r"\(\s*M\s*,\s*m\s*\)", re.IGNORECASE)

# Confidence helpers
STRONG_PROGRAM_TOKEN_RE = re.compile(r"\b(major|minor|department|program|certificate|concentration|degree)\b", re.IGNORECASE)
DEGREE_TOKEN_RE = re.compile(r"\b(b\.?a\.?|b\.?s\.?|m\.?a\.?|m\.?s\.?|ph\.?d\.?)\b", re.IGNORECASE)
DEGREE_PHRASE_RE = re.compile(
    r"\b(ba|bs|b\.a\.|b\.s\.|ma|ms|m\.a\.|m\.s\.|phd|ph\.d\.)\s+in\b|\bbachelor\s+of\s+(arts|science)\s+in\b|\bmaster\s+of\s+(arts|science)\s+in\b",
    re.IGNORECASE,
)
ADMIN_MARKERS_RE = re.compile(
    r"\b("
    r"curriculum|requirements|electives|4-year\s+plan|transfer\s+plan|catalog|degree\s+type|research\s+guide|guide"
    r"|prereq(?:uisite)?s?"
    r"|open\s+to\s+students"
    r"|have\s+declared"
    r"|program\s+of\s+study"
    r"|senior\s+priority"
    r")\b",
    re.IGNORECASE,
)
STORY_MARKERS_RE = re.compile(r"\b(reads\s+into|named|talking\s+with|studying|honoring|why\b)\b|\?", re.IGNORECASE)
PROFILE_DASH_RE = re.compile(r"\b[A-Z][a-z]+\s+[A-Z][a-z]+\b\s*[-–—]\s*", re.UNICODE)
# Patch B: Allow dash-degree suffixes like "X - Minor" through profile dash penalty
DASH_DEGREE_SUFFIX_RE = re.compile(r"[-–—]\s*(major|minor|certificate|concentration)\b", re.IGNORECASE)

BARE_STUDIES_PROGRAM_RE = re.compile(r"\bstudies\b$", re.IGNORECASE)

# =========================
# Helpers
# =========================
def _try_mojibake_repair(s: str) -> str:
    """
    Attempt to repair common UTF-8 bytes mis-decoded as cp1252/latin1.
    This is a simpler/general alternative to huge find+replace lists.
    """
    if s is None:
        return ""
    s = str(s)

    # quick heuristic: only attempt if it looks like mojibake
    if not any(x in s for x in ["‚Ä", "Ã", "Â", "�"]):
        return s

    # try cp1252 -> utf-8
    for enc in ("cp1252", "latin1"):
        try:
            repaired = s.encode(enc, errors="ignore").decode("utf-8", errors="ignore")
            if repaired and (repaired != s) and ("�" not in repaired):
                return repaired
        except Exception:
            pass
    return s

def normalize_text(s: str) -> str:
    """General cleaning: mojibake repair + unicode normalization + whitespace + HTML unescape."""
    if s is None:
        return ""
    s = str(s)

    s = html.unescape(s)
    s = _try_mojibake_repair(s)
    s = unicodedata.normalize("NFKC", s)

    # normalize common dash variants
    s = s.replace("\u2010", "-").replace("\u2011", "-").replace("\u2012", "-").replace("\u2013", "-").replace("\u2014", "-")
    s = s.replace("&", " and ")
    s = re.sub(r"\s+", " ", s).strip()
    return s

def _normalize_title(t: str) -> str:
    return normalize_text(t)

def _split_titles(cell) -> list[str]:
    if cell is None or (isinstance(cell, float) and pd.isna(cell)):
        return []
    s = str(cell).strip()
    if not s:
        return []
    parts = [p.strip() for p in s.split("|")]
    parts = [_normalize_title(p) for p in parts if p.strip()]
    # de-dupe while preserving order
    seen = set()
    out = []
    for p in parts:
        key = p.lower()
        if key not in seen:
            seen.add(key)
            out.append(p)
    return out

def _word_count(s: str) -> int:
    return len([w for w in re.split(r"\s+", s.strip()) if w])

def looks_like_junk_or_page_text(title: str) -> bool:
    t = _normalize_title(title)
    t_low = t.lower()

    if YEAR_PATTERN.search(t_low) or CLASS_YEAR_PATTERN.search(t_low):
        return True

    for p in JUNK_PHRASES:
        if p in t_low:
            return True

    # long sentence-like strings without strong program tokens
    if _word_count(t_low) > 15:
        if (not any(tok in t_low for tok in PROGRAM_TOKENS)) and ("studies" not in t_low):
            return True

    return False

def canonicalize_program_title(title: str) -> str:
    """
    Canonicalize to a 'program label' where possible, collapsing variants:
      - Department of X Studies -> X Studies
      - X Studies Major -> X Studies
      - X Studies Minor -> X Studies
      - Drop trailing parenthetical like (ABD) or (M, m)
      - Preserve trailing parenthetical tracks like (Pan African Studies)
    """
    t = _normalize_title(title)

    # Strip leading navigation/page prefixes
    t = re.sub(
        r"^\s*(academic\s+departments\s+and\s+programs|majors\s+and\s+minors|majors|minors|programs)\b[:\s-]*",
        "",
        t,
        flags=re.IGNORECASE,
    ).strip()
    t = re.sub(r"^\s*(check\s+out|view|welcome\s+to|about)\b[:\s-]*", "", t, flags=re.IGNORECASE).strip()

    # --- Parentheticals ---
    # Keep (M, m) evidence for program confidence (handled upstream), but remove it from the canonical label.
    t = re.sub(r"\s*\(\s*M\s*,\s*m\s*\)\s*$", "", t, flags=re.IGNORECASE).strip()

    # Preserve trailing parenthetical "tracks" (e.g., "Ethnic Studies (Pan African Studies)")
    # but still drop short acronym parentheticals like (ABD).
    m_track = re.search(r"\s*\(([^)]{2,120})\)\s*$", t)
    if m_track:
        track_txt = m_track.group(1).strip()
        track_low = track_txt.lower()

        # Drop common acronym-only tags like (ABD), (AFST), etc.
        if re.fullmatch(r"[A-Z]{2,6}", track_txt):
            t = re.sub(r"\s*\((?:[A-Z]{2,6})\)\s*$", "", t).strip()
        else:
            # Keep parenthetical if it looks like an academic track/specialization that affects bucketing.
            # (We keep this permissive: any in-scope bucket signal OR contains 'studies'.)
            track_has_bucket_signal = False
            for b in ["black", "africana", "mena", "african", "minority", "ethnic", "race"]:
                for pat in BUCKET_PATTERNS.get(b, []):
                    if re.search(pat, track_low, flags=re.IGNORECASE):
                        track_has_bucket_signal = True
                        break
                if track_has_bucket_signal:
                    break

            if ("studies" in track_low) or track_has_bucket_signal:
                # Keep the parenthetical track (no change)
                pass
            else:
                # Otherwise, drop the trailing parenthetical as a last resort cleanup.
                t = re.sub(r"\s*\((?:[^)]*)\)\s*$", "", t).strip()
    else:
        # If no parenthetical, no action needed
        pass

    # drop obvious nav suffixes
    t = re.sub(r"\s+(home|overview)\b", "", t, flags=re.IGNORECASE).strip()

    # normalize common prefixes
    t = re.sub(r"^\s*department\s+of\s+", "", t, flags=re.IGNORECASE).strip()
    t = re.sub(r"^\s*the\s+department\s+of\s+", "", t, flags=re.IGNORECASE).strip()
    t = re.sub(r"^\s*the\s+", "", t, flags=re.IGNORECASE).strip()

    # Strip leading degree phrases when they are just wrappers around a program title
    # (preserves any trailing parenthetical track, e.g., "Ethnic Studies (Pan African Studies)")
    t = re.sub(r"^\s*(ba|bs|b\.a\.|b\.s\.|ma|ms|m\.a\.|m\.s\.|phd|ph\.d\.)\s+in\s+", "", t, flags=re.IGNORECASE).strip()
    t = re.sub(r"^\s*bachelor\s+of\s+(arts|science)\s+in\s+", "", t, flags=re.IGNORECASE).strip()
    t = re.sub(r"^\s*master\s+of\s+(arts|science)\s+in\s+", "", t, flags=re.IGNORECASE).strip()
    t = re.sub(r"^\s*master\s+of\s+arts\s+in\s+", "", t, flags=re.IGNORECASE).strip()

    # Normalize some very common standalone variants
    # (We keep this small and conservative: only normalize when the intent is extremely clear.)
    t_low = t.lower().strip()
    if t_low == "africana":
        t = "Africana Studies"
        t_low = t.lower()

    # Remove trailing degree-structure tokens
    t = re.sub(r"\s+\bmajor\s+and\s+minor(s)?\b$", "", t, flags=re.IGNORECASE).strip()
    t = re.sub(r"\s+\b(major|minor|program)\b$", "", t, flags=re.IGNORECASE).strip()

    # If this is a combined title with multiple study areas, prefer the in-scope anchor program label
    # NOTE: do NOT collapse comma-joined labels like "African, Black and Caribbean Studies".
    t_low = t.lower()
    if " and " in t_low:
        # Patch C: prevent MENA titles like "Middle Eastern and North African Studies" from
        # accidentally collapsing to "African Studies" due to the substring "African Studies".
        if any(term in t_low for term in MENA_TERMS) and ("studies" in t_low):
            # Canonicalize to a consistent MENA label
            if "middle eastern" in t_low:
                t = "Middle Eastern and North African Studies"
            elif "middle east" in t_low:
                t = "Middle East and North Africa Studies"
            elif "north african" in t_low:
                # fall back to the more common combined label
                t = "Middle Eastern and North African Studies"
        # Patch E: keep "Race and Ethnic Studies" as a distinct program label (do NOT collapse to "Ethnic Studies")
        elif re.search(r"\brace\s+and\s+ethnic\s+studies\b", t_low, flags=re.IGNORECASE):
            t = "Race and Ethnic Studies"
        elif "ethnic studies" in t_low:
            t = "Ethnic Studies"
        elif "black studies" in t_low:
            t = "Black Studies"
        elif "africana studies" in t_low:
            t = "Africana Studies"
        elif "african and african american studies" in t_low:
            t = "African and African American Studies"
        elif "african studies" in t_low and ("african american" not in t_low) and ("african-american" not in t_low):
            t = "African Studies"

    # Explicit anchor for "African, Black and Caribbean Studies" style labels
    if re.search(r"\bafrican\s*,\s*black\s+and\s+caribbean\s+studies\b", t.lower(), flags=re.IGNORECASE):
        t = "African, Black and Caribbean Studies"

    # collapse duplicated phrase like "African and African Diaspora Studies African and African Diaspora Studies"
    words = t.split()
    if len(words) >= 6:
        mid = len(words) // 2
        if " ".join(words[:mid]).lower() == " ".join(words[mid:]).lower():
            t = " ".join(words[:mid]).strip()

    # Trim trailing punctuation artifacts (commas, colons, etc.)
    t = re.sub(r"[\s,:;]+$", "", t).strip()

    return t

def bucket_title(title: str) -> str:
    t_norm = _normalize_title(title)
    t_low = t_norm.lower()

    for fp in BLACK_FALSE_POSITIVES:
        if fp in t_low:
            return "other"

    if not t_low:
        return "other"

    # Explicit overrides for common joint/compound labels
    if re.search(r"\bafrican\s*,?\s*black\s+and\s+caribbean\s+studies\b", t_low, flags=re.IGNORECASE):
        return "black"
    if re.search(r"\bafrican\s+and\s+african[-\s]?american\s+studies\b", t_low, flags=re.IGNORECASE):
        return "black"
    # Patch E: "Race and Ethnic Studies" should be treated as a race-bucket program (race > ethnic for this phrase)
    if re.search(r"\brace\s+and\s+ethnic\s+studies\b", t_low, flags=re.IGNORECASE):
        return "race"

    for bucket in BUCKET_ORDER:
        if bucket == "other":
            continue
        for pat in BUCKET_PATTERNS.get(bucket, []):
            if re.search(pat, t_low, flags=re.IGNORECASE):
                if bucket == "african":
                    if re.search(r"\bafrican[-\s]?american\b", t_low) or re.search(r"\bafricana\b", t_low):
                        continue
                    if any(term in t_low for term in MENA_TERMS):
                        continue
                return bucket

    return "other"

def program_confidence(title: str, source: str = "") -> tuple[int, str]:
    """Return (0-100 confidence, reason_flags)."""
    t = _normalize_title(title)
    t_low = t.lower()
    flags = []
    score = 0
    wc = _word_count(t_low)

    # Patch D (revised): CV taxonomy label should be treated as a target program.
    # This label often lacks degree/program tokens but should be included as a program in the minority bucket.
    if source == "cv" and re.search(
        r"^ethnic\s*,\s*cultural\s+minority\s*,\s*and\s+gender\s+studies\s*,\s*other\b",
        t_low,
        flags=re.IGNORECASE,
    ):
        score += 70
        flags.append("cv_taxonomy_target")

    # Positive evidence
    if STRONG_PROGRAM_TOKEN_RE.search(t_low):
        score += 60
        flags.append("strong_token")

    if DEGREE_TOKEN_RE.search(t_low):
        score += 40
        flags.append("degree_token")

    if DEGREE_PHRASE_RE.search(t_low):
        score += 60
        flags.append("degree_phrase")

    if MM_SHORTHAND_RE.search(t):
        score += 40
        flags.append("major_minor_shorthand")

    # Bare 'X Studies' program name (only boost when it appears in-scope)
    if BARE_STUDIES_PROGRAM_RE.search(t_low) and wc <= 6:
        bkt = bucket_title(t)
        if bkt != "other":
            score += 60
            flags.append("bare_studies_name")

    # Anchor program labels even when embedded (helps joint titles like "Ethnic Studies and ...")
    if any(x in t_low for x in [
        "ethnic studies",
        "black studies",
        "africana studies",
        "african studies",
        "african and african american studies",
    ]):
        score += 20
        flags.append("anchor_phrase")
    # Compound titles that clearly contain an anchor program label (common on catalog pages)
    # e.g. "Ethnic Studies and Women's, Gender, and Sexuality Studies"
    if (" and " in t_low) and any(x in t_low for x in [
        "ethnic studies",
        "black studies",
        "africana studies",
        "african studies",
        "african and african american studies",
    ]):
        # Only boost if it doesn't look like admin/course/story
        if (not ADMIN_MARKERS_RE.search(t_low)) and (not STORY_MARKERS_RE.search(t_low)) and (not DESCRIPTOR_NONPROGRAM_RE.search(t_low)):
            score += 40
            flags.append("compound_anchor")
            
    # Bucket match is weak evidence
    # IMPORTANT: bucket_match should reflect the FINAL bucket decision (bucket_title),
    # not the first regex pattern hit. This keeps the reason string consistent with
    # overrides like Patch E (e.g., "Race and Ethnic Studies" -> race).
    bkt = bucket_title(t)
    if bkt != "other":
        score += 10
        flags.append(f"bucket_match:{bkt}")

    # CV source bonus: taxonomy labels often omit tokens
    if source == "cv" and ("/" in t or ("studies" in t_low and wc <= 8)):
        score += 20
        flags.append("cv_bonus")

    # Penalties
    if ADMIN_MARKERS_RE.search(t_low):
        score -= 60
        flags.append("admin_marker")

    if STORY_MARKERS_RE.search(t_low):
        score -= 60
        flags.append("story_marker")

    # Profile-style pages often look like "Firstname Lastname - Africana Studies".
    # But allow real program markers like "X - Minor" / "X - Major" to pass through.
    if PROFILE_DASH_RE.search(t) and (not DASH_DEGREE_SUFFIX_RE.search(t)):
        score -= 60
        flags.append("profile_dash")

    # Descriptor/support pages that are not themselves programs
    if DESCRIPTOR_NONPROGRAM_RE.search(t_low):
        score -= 80
        flags.append("descriptor_nonprogram")

    # Course-like, unless it has strong program tokens or (M,m) shorthand
    if (COURSE_PREFIX_RE.search(t_low) or any(p in t_low for p in COURSE_PHRASES)):
        if (not STRONG_PROGRAM_TOKEN_RE.search(t_low)) and (not MM_SHORTHAND_RE.search(t)):
            score -= 80
            flags.append("course_like")

    if wc > 12 and not STRONG_PROGRAM_TOKEN_RE.search(t_low) and not DEGREE_TOKEN_RE.search(t_low):
        score -= 50
        flags.append("sentence_like")

    score = max(0, min(100, score))
    return score, "|".join(flags)

def classify_title_kind(title: str, source: str = "") -> str:
    """
    Classify extracted strings; we will only print those that are 'program' in wide outputs.
    We still store other kinds in long outputs.
    """
    t = _normalize_title(title)
    t_low = t.lower()

    if not t_low:
        return "junk"

    # Patch D (revised): Treat the specific CV taxonomy label as a target program.
    # Ensures it is included as a program even though it may omit program/degree tokens.
    if source == "cv" and re.search(
        r"^ethnic\s*,\s*cultural\s+minority\s*,\s*and\s+gender\s+studies\s*,\s*other\b",
        t_low,
        flags=re.IGNORECASE,
    ):
        return "program"

    # Hard junk/page text
    if looks_like_junk_or_page_text(t_low):
        return "junk"

    # Guard against black-related false positives
    if any(fp in t_low for fp in BLACK_FALSE_POSITIVES):
        # unless explicitly Black Studies / African-American studies
        if ("black studies" not in t_low) and ("african-american" not in t_low) and ("african american" not in t_low):
            return "junk"

    # Descriptor/support pages that are not themselves programs
    if DESCRIPTOR_NONPROGRAM_RE.search(t_low):
        return "program_signal"

    # Department/unit pages (real signal but not a degree program by themselves)
    if re.search(r"\bdepartment\b", t_low) or re.search(r"^department\s+of\b", t_low):
        # If it also clearly specifies a degree (e.g., BA/BS/MA/PhD), keep it eligible as a program
        if DEGREE_TOKEN_RE.search(t_low) or DEGREE_PHRASE_RE.search(t_low):
            pass
        else:
            return "department_unit"

    # Course number
    if re.search(r"\b\d{3,4}\b", t_low):
        return "course"

    # Centers/institutes
    if any(x in t_low for x in ["center", "centre", "institute", "initiative", "lab", "laboratory", "research center", "research centre"]):
        return "center_institute"

    # Committees/councils/admin
    if any(x in t_low for x in ["committee", "council", "office", "division", "board"]):
        return "committee_admin"

    # Event/news-like
    if any(v in t_low for v in TRIP_STORY_VERBS):
        return "event_news"
    if any(c in t_low for c in COUNTRY_TERMS) and (not any(tok in t_low for tok in PROGRAM_TOKENS)) and (not MM_SHORTHAND_RE.search(t)):
        return "event_news"

    # Course-like prefixes/phrases without strong program tokens and without (M, m) shorthand => course
    if (COURSE_PREFIX_RE.search(t_low) or any(p in t_low for p in COURSE_PHRASES)):
        if (not any(tok in t_low for tok in PROGRAM_TOKENS)) and (not MM_SHORTHAND_RE.search(t)):
            return "course"

    # Confidence-based decision
    conf, _ = program_confidence(t, source=source)

    if conf >= 70:
        return "program"
    if conf >= 40:
        return "maybe_program"
    return "other_nonprogram"

def format_bucket_cell(items: list[str], source: str) -> str:
    if not items:
        return ""
    return f"{source}:" + "|".join(items)

def merge_sources_cell(crawl_items: list[str], cv_items: list[str]) -> str:
    parts = []
    if crawl_items:
        parts.append(format_bucket_cell(crawl_items, "crawl"))
    if cv_items:
        parts.append(format_bucket_cell(cv_items, "cv"))
    return " || ".join(parts)

def main() -> None:
    df = pd.read_csv(INPUT_CSV, dtype=str)
    # Ensure row_index is always a simple 0..N-1 integer index (prevents tuple/MultiIndex issues)
    df = df.reset_index(drop=True)

    for col in [CRAWL_TITLES_COL, CV_TITLES_COL]:
        if col not in df.columns:
            df[col] = ""

    # normalize mojibake in reference program name col (and extracted title cols)
    if REF_PROGRAM_NAME_COL in df.columns:
        df[REF_PROGRAM_NAME_COL] = df[REF_PROGRAM_NAME_COL].fillna("").map(normalize_text)

    for c in [CRAWL_TITLES_COL, CV_TITLES_COL]:
        df[c] = df[c].fillna("").map(normalize_text)

    # LONG evidence table
    records = []
    for idx, row in df.iterrows():
        unitid = row.get("unitid", "")
        crawl_titles = _split_titles(row.get(CRAWL_TITLES_COL, ""))
        cv_titles    = _split_titles(row.get(CV_TITLES_COL, ""))

        for t in crawl_titles:
            conf, conf_reason = program_confidence(t, source="crawl")
            kind = classify_title_kind(t, source="crawl")
            is_prog = 1 if kind == "program" else 0
            # For maybe_program, still compute bucket/canonical so we can diagnose and promote later
            # Always compute bucket/canonical if it matches a bucket OR is a real signal/program candidate.
            bucket_guess = bucket_title(t)
            if (kind in ("program", "maybe_program")) or (kind in REAL_NONPROGRAM_KINDS) or (bucket_guess != "other"):
                bucket = bucket_guess
                canonical = canonicalize_program_title(t)
            else:
                bucket = ""
                canonical = ""
            records.append({
                "row_index": int(idx),
                "unitid": unitid,
                "source": "crawl",
                "raw_title": t,
                "title_kind": kind,
                "is_program_title": is_prog,
                "bucket": bucket,
                "canonical_title": canonical,
                "program_conf": conf,
                "program_conf_reason": conf_reason,
            })

        for t in cv_titles:
            conf, conf_reason = program_confidence(t, source="cv")
            kind = classify_title_kind(t, source="cv")
            is_prog = 1 if kind == "program" else 0
            # For maybe_program, still compute bucket/canonical so we can diagnose and promote later
            # Always compute bucket/canonical if it matches a bucket OR is a real signal/program candidate.
            bucket_guess = bucket_title(t)
            if (kind in ("program", "maybe_program")) or (kind in REAL_NONPROGRAM_KINDS) or (bucket_guess != "other"):
                bucket = bucket_guess
                canonical = canonicalize_program_title(t)
            else:
                bucket = ""
                canonical = ""
            records.append({
                "row_index": int(idx),
                "unitid": unitid,
                "source": "cv",
                "raw_title": t,
                "title_kind": kind,
                "is_program_title": is_prog,
                "bucket": bucket,
                "canonical_title": canonical,
                "program_conf": conf,
                "program_conf_reason": conf_reason,
            })

    long_df = pd.DataFrame.from_records(records)
    if not long_df.empty and "row_index" in long_df.columns:
        # Hardening: keep row_index a plain integer column
        long_df["row_index"] = pd.to_numeric(long_df["row_index"], errors="coerce").fillna(-1).astype(int)

    # Ensure REAL_NONPROGRAM_KINDS (especially departments) always get bucket/canonical filled
    # even if upstream logic left them blank for any reason.
    if not long_df.empty:
        _np_mask = long_df["title_kind"].isin(REAL_NONPROGRAM_KINDS)
        if _np_mask.any():
            # Fill bucket when missing/blank
            _bucket_blank = long_df.loc[_np_mask, "bucket"].astype(str).str.strip().eq("")
            if _bucket_blank.any():
                long_df.loc[_np_mask & _bucket_blank, "bucket"] = long_df.loc[_np_mask & _bucket_blank, "raw_title"].map(bucket_title)

            # Fill canonical_title when missing/blank
            _canon_blank = long_df.loc[_np_mask, "canonical_title"].astype(str).str.strip().eq("")
            if _canon_blank.any():
                long_df.loc[_np_mask & _canon_blank, "canonical_title"] = long_df.loc[_np_mask & _canon_blank, "raw_title"].map(canonicalize_program_title)

            # As a final fallback, normalize any remaining empty buckets to "other"
            long_df.loc[_np_mask & long_df["bucket"].astype(str).str.strip().eq(""), "bucket"] = "other"

    # Signals-only (real but not programs) for downstream
    long_signals_df = long_df[long_df["title_kind"].isin(REAL_NONPROGRAM_KINDS)].copy() if not long_df.empty else pd.DataFrame()

    # Hardening: ensure real non-program signals ALWAYS have a bucket + canonical_title
    # (prevents blanks from leaking into long outputs and keeps wide signals consistent)
    if not long_signals_df.empty:
        # bucket fallback
        long_signals_df["bucket"] = long_signals_df["bucket"].astype(str).str.strip()
        _b_blank = long_signals_df["bucket"].eq("") | long_signals_df["bucket"].isna()
        if _b_blank.any():
            long_signals_df.loc[_b_blank, "bucket"] = long_signals_df.loc[_b_blank, "raw_title"].map(bucket_title)
        long_signals_df.loc[long_signals_df["bucket"].astype(str).str.strip().eq(""), "bucket"] = "other"

        # canonical fallback
        long_signals_df["canonical_title"] = long_signals_df["canonical_title"].astype(str).str.strip()
        _c_blank = long_signals_df["canonical_title"].eq("") | long_signals_df["canonical_title"].isna()
        if _c_blank.any():
            long_signals_df.loc[_c_blank, "canonical_title"] = long_signals_df.loc[_c_blank, "raw_title"].map(canonicalize_program_title)

    # Program-only view for inventory and pivot
    long_prog = long_df[(long_df["is_program_title"] == 1) & (long_df["bucket"].astype(str).str.len() > 0)].copy()

    # =========================
    # Program bucket provenance (for long summary output)
    # =========================
    if not long_prog.empty:
        bucket_src = (
            long_prog
            .groupby(["row_index", "unitid", "bucket"], dropna=False)["source"]
            .apply(lambda s: "|".join(sorted(set([str(x) for x in s if pd.notna(x) and str(x) != ""]))))
            .reset_index()
            .rename(columns={"source": "sources_in_key"})
        )
        bucket_src["has_crawl_in_key"] = bucket_src["sources_in_key"].str.contains(r"\bcrawl\b", regex=True).astype(int)
        bucket_src["has_cv_in_key"]    = bucket_src["sources_in_key"].str.contains(r"\bcv\b", regex=True).astype(int)
        bucket_src["has_both_in_key"]  = ((bucket_src["has_crawl_in_key"] == 1) & (bucket_src["has_cv_in_key"] == 1)).astype(int)
        bucket_src["key_type"] = "bucket"
        bucket_src = bucket_src.rename(columns={"bucket": "key"})

        # merge provenance back to program rows
        long_prog = long_prog.merge(
            bucket_src[["row_index", "unitid", "key", "sources_in_key", "has_crawl_in_key", "has_cv_in_key", "has_both_in_key"]],
            left_on=["row_index", "unitid", "bucket"],
            right_on=["row_index", "unitid", "key"],
            how="left",
        ).drop(columns=["key"], errors="ignore")
    else:
        bucket_src = pd.DataFrame(columns=[
            "row_index", "unitid", "key", "sources_in_key",
            "has_crawl_in_key", "has_cv_in_key", "has_both_in_key", "key_type"
        ])

    # Non-program provenance (kept only in long summary output)
    long_nonprog = long_df[long_df["is_program_title"] == 0].copy()
    if not long_nonprog.empty:
        # Ensure bucket is never blank for the real non-program kinds we track
        _np_mask2 = long_nonprog["title_kind"].isin(REAL_NONPROGRAM_KINDS)
        if _np_mask2.any():
            long_nonprog["bucket"] = long_nonprog["bucket"].astype(str).str.strip()
            _b_blank2 = _np_mask2 & (long_nonprog["bucket"].eq("") | long_nonprog["bucket"].isna())
            if _b_blank2.any():
                long_nonprog.loc[_b_blank2, "bucket"] = long_nonprog.loc[_b_blank2, "raw_title"].map(bucket_title)
            long_nonprog.loc[_np_mask2 & long_nonprog["bucket"].astype(str).str.strip().eq(""), "bucket"] = "other"

        signal_src = (
            long_nonprog[long_nonprog["title_kind"].isin(REAL_NONPROGRAM_KINDS)]
            .groupby(["row_index", "unitid", "title_kind", "bucket"], dropna=False)["source"]
            .apply(lambda s: "|".join(sorted(set([str(x) for x in s if pd.notna(x) and str(x) != ""]))))
            .reset_index()
            .rename(columns={"title_kind": "key", "source": "sources_in_key"})
        )
        signal_src["has_crawl_in_key"] = signal_src["sources_in_key"].str.contains(r"\bcrawl\b", regex=True).astype(int)
        signal_src["has_cv_in_key"]    = signal_src["sources_in_key"].str.contains(r"\bcv\b", regex=True).astype(int)
        signal_src["has_both_in_key"]  = ((signal_src["has_crawl_in_key"] == 1) & (signal_src["has_cv_in_key"] == 1)).astype(int)
        signal_src["key_type"] = "title_kind"
    else:
        signal_src = pd.DataFrame(columns=[
            "row_index", "unitid", "bucket", "key", "sources_in_key",
            "has_crawl_in_key", "has_cv_in_key", "has_both_in_key", "key_type"
        ])

    long_bucket_df = pd.concat([bucket_src, signal_src], ignore_index=True)

    # =========================
    # Build program inventory tables (clean)
    # =========================
    if not long_prog.empty:
        # --- Inventory-specific shaping ---
        # 1) If a unit has a clear anchor program label (e.g., "Black Studies"), fold course/admin-like
        #    spillover titles into that anchor as supporting evidence instead of separate canonical programs.
        # 2) Expand sentences like "X Studies is offered as a major, minor, and concentration." into
        #    three inventory rows (major/minor/concentration) for program accounting.

        def _is_demotable_as_evidence(raw: str) -> bool:
            t = _normalize_title(raw)
            t_low = t.lower()
            if not t_low:
                return True
            if DESCRIPTOR_NONPROGRAM_RE.search(t_low):
                return True
            if ADMIN_MARKERS_RE.search(t_low):
                return True
            if STORY_MARKERS_RE.search(t_low):
                return True
            if PROFILE_DASH_RE.search(t):
                return True
            if (COURSE_PREFIX_RE.search(t_low) or any(p in t_low for p in COURSE_PHRASES)):
                return True
            return False

        ANCHOR_CANONICALS = {
            "Black Studies",
            "Africana Studies",
            "African Studies",
            "Ethnic Studies",
            "African and African American Studies",
            "African-American/Black Studies",
        }

        # Work on a copy so we don't affect downstream wide pivot behavior
        inv_prog = long_prog.copy()

        # Anchor folding within (unitid, bucket, source)
        inv_prog["_is_anchor"] = inv_prog["canonical_title"].isin(ANCHOR_CANONICALS)
        inv_prog["_demote"] = inv_prog["raw_title"].map(_is_demotable_as_evidence)

        def _fold_to_anchor(g: pd.DataFrame) -> pd.DataFrame:
            anchors = g[g["_is_anchor"]]
            if anchors.empty:
                return g
            # Prefer the most common anchor title; if tie, the first
            anchor_title = anchors["canonical_title"].value_counts().index[0]
            g.loc[g["_demote"], "canonical_title"] = anchor_title
            return g

        inv_prog = (
            inv_prog
            .groupby(["unitid", "bucket", "source"], dropna=False, group_keys=False)
            .apply(_fold_to_anchor)
        )

        # Sentence expansion for inventory only
        # Example: "Ethnic Studies is offered as a major, minor, and concentration." => 3 inventory rows
        def _expand_offered_as(g: pd.DataFrame) -> pd.DataFrame:
            out_rows = []
            for _, r in g.iterrows():
                raw = str(r.get("raw_title", ""))
                raw_low = raw.lower()
                if "is offered as" in raw_low and "major" in raw_low and "minor" in raw_low and "concentration" in raw_low:
                    base = str(r.get("canonical_title", "")).strip() or str(r.get("raw_title", "")).strip()
                    # Use inventory-specific titles that preserve the three-program signal
                    for suffix in ("Major", "Minor", "Concentration"):
                        r2 = r.copy()
                        r2["canonical_title"] = f"{base} ({suffix})"
                        out_rows.append(r2)
                else:
                    out_rows.append(r)
            return pd.DataFrame(out_rows)

        inv_prog = (
            inv_prog
            .groupby(["unitid", "bucket", "source"], dropna=False, group_keys=False)
            .apply(_expand_offered_as)
        )

        # Build inventory tables
        long_programs_df = (
            inv_prog
            .groupby(["unitid", "source", "bucket", "canonical_title"], dropna=False)
            .agg(
                supporting_raw_titles=("raw_title", lambda xs: "|".join(sorted(set([str(x) for x in xs if str(x).strip()])))),
                program_conf_max=("program_conf", "max"),
                program_conf_reasons=("program_conf_reason", lambda xs: "|".join(sorted(set([str(x) for x in xs if str(x).strip()])))),
            )
            .reset_index()
        )

        long_programs_agg = (
            long_programs_df
            .groupby(["unitid", "bucket", "canonical_title"], dropna=False)
            .agg(
                sources=("source", lambda xs: "|".join(sorted(set(xs)))),
                program_conf_max=("program_conf_max", "max"),
            )
            .reset_index()
        )
        long_programs_agg["has_both_sources"] = long_programs_agg["sources"].str.contains("crawl") & long_programs_agg["sources"].str.contains("cv")
        long_programs_agg["program_conf_final"] = (long_programs_agg["program_conf_max"] + long_programs_agg["has_both_sources"].astype(int) * 10).clip(0, 100)
    else:
        long_programs_df = pd.DataFrame(columns=["unitid", "source", "bucket", "canonical_title", "supporting_raw_titles", "program_conf_max", "program_conf_reasons"])
        long_programs_agg = pd.DataFrame(columns=["unitid", "bucket", "canonical_title", "sources", "program_conf_max", "has_both_sources", "program_conf_final"])

    # =========================
    # Pivot back to WIDE: one column per bucket with provenance
    # =========================
    for b in BUCKET_ORDER:
        df[f"program_bucket__{b}"] = ""
        df[f"program_bucket__{b}__crawl"] = ""
        df[f"program_bucket__{b}__cv"] = ""

    # NEW: wide column for real non-program signals
    df["real_nonprogram_signals"] = ""

    if not long_prog.empty:
        grouped = (
            long_prog
            .groupby(["row_index", "bucket", "source"])["canonical_title"]
            .apply(list)
            .reset_index()
        )

        for (row_index, bucket), sub in grouped.groupby(["row_index", "bucket"]):
            crawl_items = []
            cv_items = []
            for _, r in sub.iterrows():
                if r["source"] == "crawl":
                    crawl_items = r["canonical_title"]
                elif r["source"] == "cv":
                    cv_items = r["canonical_title"]

            # de-dupe in-source
            def _dedupe(xs):
                seen = set()
                out = []
                for x in xs:
                    k = str(x).lower()
                    if k not in seen and str(x).strip():
                        seen.add(k)
                        out.append(x)
                return out

            crawl_items = _dedupe(crawl_items)
            cv_items = _dedupe(cv_items)

            df.at[row_index, f"program_bucket__{bucket}__crawl"] = "|".join(crawl_items) if crawl_items else ""
            df.at[row_index, f"program_bucket__{bucket}__cv"]    = "|".join(cv_items) if cv_items else ""
            df.at[row_index, f"program_bucket__{bucket}"]        = merge_sources_cell(crawl_items, cv_items)

    # Fill wide real_nonprogram_signals with provenance (centers/committees)
    if not long_signals_df.empty:
        sig_grouped = (
            long_signals_df
            .groupby(["row_index", "title_kind", "bucket", "source"])["canonical_title"]
            .apply(list)
            .reset_index()
        )

        for row_index, sub in sig_grouped.groupby("row_index"):
            parts = []
            for _, r in sub.iterrows():
                kind = r["title_kind"]
                bkt = r["bucket"] if str(r.get("bucket", "")).strip() else "other"
                src = r["source"]
                titles = r["canonical_title"] if isinstance(r["canonical_title"], list) else [r["canonical_title"]]

                # de-dupe
                seen = set()
                titles2 = []
                for x in titles:
                    x = str(x).strip()
                    if not x:
                        continue
                    k = x.lower()
                    if k not in seen:
                        seen.add(k)
                        titles2.append(x)

                parts.append(f"{kind}:{bkt}:{src}:" + "|".join(titles2))
            df.at[row_index, "real_nonprogram_signals"] = " || ".join(parts)

    # Bucket hit summary
    def _bucket_hit_list(row) -> str:
        hits = []
        for b in BUCKET_ORDER:
            if row.get(f"program_bucket__{b}", ""):
                hits.append(b)
        return "|".join(hits)

    df["program_buckets_hit"] = df.apply(_bucket_hit_list, axis=1)

    # =========================
    # Save
    # =========================
    df.to_csv(OUTPUT_CSV, index=False)
    long_df.to_csv(OUTPUT_LONG_CSV, index=False)
    long_bucket_df.to_csv(OUTPUT_LONG_BUCKET_CSV, index=False)

    long_programs_df.to_csv(OUTPUT_LONG_PROGRAMS_CSV, index=False)
    long_programs_agg.to_csv(OUTPUT_LONG_PROGRAMS_AGG_CSV, index=False)
    long_signals_df.to_csv(OUTPUT_LONG_SIGNALS_CSV, index=False)

    print("Wrote:")
    print(" -", OUTPUT_CSV)
    print(" -", OUTPUT_LONG_CSV)
    print(" -", OUTPUT_LONG_BUCKET_CSV)
    print(" -", OUTPUT_LONG_PROGRAMS_CSV)
    print(" -", OUTPUT_LONG_PROGRAMS_AGG_CSV)
    print(" -", OUTPUT_LONG_SIGNALS_CSV)

if __name__ == "__main__":
    main()

/var/folders/0_/vt5npj3x5wq9q_4bzlj28tqw0000gn/T/ipykernel_13105/781875674.py:873: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  inv_prog
/var/folders/0_/vt5npj3x5wq9q_4bzlj28tqw0000gn/T/ipykernel_13105/781875674.py:897: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  inv_prog


Wrote:
 - ace_unitid_merge__ace_x_2013comp__webscrape__v15simple__bucketed_programs.csv
 - ace_unitid_merge__ace_x_2013comp__webscrape__v15simple__bucketed_programs__long.csv
 - ace_unitid_merge__ace_x_2013comp__webscrape__v15simple__bucketed_programs__long_bucket_summary.csv
 - ace_unitid_merge__ace_x_2013comp__webscrape__v15simple__bucketed_programs__long_programs.csv
 - ace_unitid_merge__ace_x_2013comp__webscrape__v15simple__bucketed_programs__long_programs_agg.csv
 - ace_unitid_merge__ace_x_2013comp__webscrape__v15simple__bucketed_programs__long_signals.csv


In [36]:
#!/usr/bin/env python3
"""2013 vs Current Program/Signal Matching

DATA DICTIONARY (output columns)
===============================
This script compares each institution's single 2013 program name against *current* discovered titles
from multiple sources (crawl + CollegeVine), within the same `unitid`.

Input expectations (wide CSV)
-----------------------------
Required columns:
  - unitid
  - name
  - 2013_program_name
  - program_titles_found                       (pipe/semicolon/newline separated list; mostly crawl)
  - college_vine_program_titles_found          (pipe/semicolon/newline separated list; CV)

Optional columns (used if present):
  - real_nonprogram_signals                    (pipe/semicolon/newline separated; may be prefixed with `crawl:` / `cv:`)

Output columns
--------------
1)  match_2013__best_title
    - The matched candidate title selected by the match ladder. Empty if no match.

2)  match_2013__best_source
    - Where the best_title came from: crawl | cv | signal | both | unknown.
      `both` occurs when the same title appears in both crawl and cv candidate pools.

3)  match_2013__best_kind
    - Candidate kind: program | signal. (Signals come from `real_nonprogram_signals` or titles that look like
      departments/centers/etc. and are treated as signals when matching.)

4)  match_2013__match_level
    - One of: strict_raw | strict_canonical | fuzzy_raw | fuzzy_canonical | fuzzy_syn | related_credential | family_rescue | related_domain_backstop | NO_MATCH

5)  match_2013__match_score
    - Score for fuzzy matches (0-1). For strict matches, 1.0. For NO_MATCH, 0.

6)  match_2013__detail
    - Human-readable detail about the selected match (best pair + score).

7)  match_2013__is_signal_marker_in_2013
    - 1 if 2013_program_name contains an explicit signal marker (department/center/institute/committee/etc.), else 0.

8)  discovered__program_titles__crawl
    - Canonicalized list of crawl candidates (deduped, pipe-joined).

9)  discovered__program_titles__cv
    - Canonicalized list of CV candidates (deduped, pipe-joined).

10) discovered__signal_titles
    - Canonicalized list of signal candidates (deduped, pipe-joined).

11) discovered__all_titles
    - All deduped titles (raw display, pipe-joined), across crawl/cv/signal.

12) discovered__new_titles_unmatched
    - Titles that were discovered (across crawl/cv/signal) but did not match 2013_program_name
      under ANY match mode (strict/raw, strict/canon, fuzzy/raw, fuzzy/canon, fuzzy/syn).

13) discovered__new_program_titles_when_best_signal
    - If best match is a signal-kind (e.g., Department), this lists program-ish titles also discovered
      (crawl/cv) that did not become the best match. Empty otherwise.

- Matching is *within unitid* and uses a ladder:
    (1) strict_raw   (normalized raw equality)
    (2) strict_canonical
    (3) fuzzy_raw
    (4) fuzzy_canonical
    (5) fuzzy_syn     (fuzzy with conservative synonym map)
    (6) related_credential (credentialed program items within domain family; stricter gating)
    (7) family_rescue (gated: direct family overlap + anchor keywords; low lexical threshold)
    (8) related_domain_backstop (best-available related match; see below)
- Default behavior prefers program-kind candidates UNLESS the 2013 string itself contains explicit signal markers.
  If 2013 looks signal-like, we prefer signal candidates when they match.
- "Approximate matching" uses max(token overlap coefficient, SequenceMatcher ratio).
- `related_domain_backstop` is used only when no other match mode succeeds; it uses domain-keyword gating plus a lower threshold to pick a best-available related match.

Usage
-----
  python 2013_current_comparison.py \
    --input ace_unitid_merge__ace_x_2013comp__webscrape__v15simple__bucketed_programs.csv

Outputs a new CSV alongside the input with suffix:
  __2013_current_matches.csv

"""

from __future__ import annotations

import argparse
import csv
import difflib
import os
import re
import sys
import unicodedata
import unittest
from dataclasses import dataclass
from typing import Dict, Iterable, List, Optional, Sequence, Set, Tuple

import pandas as pd


# ============================================================
# Normalization helpers
# ============================================================

_WS_RE = re.compile(r"\s+")


def normalize_unicode_text(s: str) -> str:
    """Normalize unicode + whitespace and strip."""
    if s is None:
        return ""
    s = str(s)
    s = unicodedata.normalize("NFKC", s)
    s = s.replace("\u00a0", " ")
    s = _WS_RE.sub(" ", s).strip()
    return s


_PUNCT_TO_SPACE = re.compile(r"[\t\n\r\f\v\u2010\u2011\u2012\u2013\u2014\u2212_/]+")


def _basic_clean(s: str) -> str:
    s = normalize_unicode_text(s)
    s = s.replace("&", " and ")
    s = _PUNCT_TO_SPACE.sub(" ", s)
    s = re.sub(r"[\"'’]", "", s)
    s = _WS_RE.sub(" ", s).strip()
    return s


def norm_title_key(s: str) -> str:
    """Strict-ish key: normalize unicode, lower, collapse whitespace, drop most punctuation."""
    s = _basic_clean(s)
    s = s.lower()
    return s


def norm_title_key_loose(s: str) -> str:
    """Loose key: strict key plus removal of some stop-words + degree noise."""
    s = norm_title_key(s)
    if not s:
        return ""

    # remove very common degree/label noise
    s = re.sub(r"\b(b\.?a\.?|b\.?s\.?|ba|bs|m\.?a\.?|m\.?s\.?|ma|ms|ph\.?d\.?|phd)\b", " ", s)
    s = re.sub(r"\b(bachelor|master|masters|doctorate|doctoral)\b", " ", s)
    s = re.sub(r"\b(major|minor|certificate|concentration|track|option|emphasis)\b", " ", s)
    s = re.sub(r"\b(in|of|and|for|the|a|an)\b", " ", s)
    s = _WS_RE.sub(" ", s).strip()
    return s


_PAREN_CONTENT = re.compile(r"\s*\([^)]*\)\s*")


def canonicalize_program_title(s: str, drop_parenthetical: bool = False) -> str:
    """Conservative canonicalization.

    - Keeps the full title by default.
    - If drop_parenthetical=True, removes parenthetical segments (useful for matching).
    """
    s = normalize_unicode_text(s)
    if not s:
        return ""

    # normalize hyphens/quotes and spacing
    s = s.replace("&", " and ")
    s = _WS_RE.sub(" ", s).strip()

    if drop_parenthetical:
        s = _PAREN_CONTENT.sub(" ", s)
        s = _WS_RE.sub(" ", s).strip()

    # remove common leading degree phrases (do NOT remove mid-title degree tokens)
    s = re.sub(r"^(ba|bs|b\.a\.|b\.s\.|ma|ms|m\.a\.|m\.s\.|master of arts|master of science|bachelor of arts|bachelor of science)\s+in\s+",
               "", s, flags=re.I)

    # strip repeated whitespace
    s = _WS_RE.sub(" ", s).strip()
    return s


# ============================================================
# Candidate splitting / parsing
# ============================================================


def _split_program_name_field(
    s: str,
    debug_added: Optional[List[str]] = None,
    split_pipes: bool = True,
    atomic_pipe_taxonomy: bool = False,
) -> List[str]:
    """Split a pipe/semicolon/newline-delimited field into items.

    Only split on commas when it looks like an actual list (>=2 commas), to avoid
    breaking names like "African, Black and Caribbean Studies".

    Recombination policy:
      - If the raw field contains `|` and yields >=3 non-empty segments, ALWAYS add:
          (a) full recombined title (segments joined by spaces)
          (b) recombined title with trailing junk segments removed (Other, Overview, etc.)

    Patch 5: atomic_pipe_taxonomy disables pipe splitting for taxonomy/category labels.
    """
    s = normalize_unicode_text(s or "")
    if not s:
        return []

    # Patch A (upgrade of Patch 5): atomic taxonomy heuristic — detect taxonomy/category labels with pipe-encoding.
    # IMPORTANT: Do NOT replace pipes with commas; commas may later be treated as list separators.
    # Instead, replace with a non-list visual delimiter and suppress comma-based splitting below.
    s_for_split = s
    _atomic_applied = False
    _suppress_comma_split = False

    # Patch A2: CV taxonomy labels often arrive comma-encoded like:
    #   "Ethnic, Cultural Minority, and Gender Studies, Other"
    # When atomic_pipe_taxonomy is enabled (CV pool), treat these as atomic and do NOT split on commas.
    if atomic_pipe_taxonomy and ("|" not in s) and (s.count(",") >= 2):
        segs_comma = [x.strip() for x in s.split(",") if x.strip()]
        if len(segs_comma) >= 3:
            last_seg = segs_comma[-1].lower()
            has_studies = ("studies" in s.lower())
            if last_seg in {"other", "overview", "general", "misc"} and has_studies:
                _suppress_comma_split = True
    if atomic_pipe_taxonomy and "|" in s:
        segs = [x.strip() for x in s.split("|") if x.strip()]
        if len(segs) >= 3:
            last_seg = segs[-1].lower()
            has_studies = any("studies" in seg.lower() for seg in segs)
            if last_seg in {"other", "overview", "general", "misc"} or has_studies:
                # treat as atomic; do not split on pipes
                s_for_split = " / ".join(segs)
                split_pipes = False
                _atomic_applied = True
                _suppress_comma_split = True

    # --- unconditional recombination when pipe fragments are present ---
    recombined_variants: List[str] = []
    if "|" in s:
        segs = [x.strip() for x in s.split("|") if x.strip()]
        if len(segs) >= 3:
            full = normalize_unicode_text(" ".join(segs))
            if full:
                recombined_variants.append(full)

            # Remove trailing junk segments like "Other", "Overview", etc.
            junk = {
                "other",
                "overview",
                "general",
                "misc",
                "programs",
                "program",
                "more",
                "learn more",
                "details",
                "information",
            }
            segs2 = list(segs)
            while segs2 and segs2[-1].strip().lower() in junk:
                segs2 = segs2[:-1]
            if segs2 and segs2 != segs:
                trimmed = normalize_unicode_text(" ".join(segs2))
                if trimmed:
                    recombined_variants.append(trimmed)

            if debug_added is not None and recombined_variants:
                # Keep debug compact: show only the created variants.
                debug_added.append(
                    f"recombined_from_pipe: {normalize_unicode_text(s)} => {', '.join(recombined_variants)}"
                )

    # Patch 5: build separator list dynamically
    seps = [";", "\n", "\r"]
    if split_pipes:
        seps = ["|"] + seps
    parts: List[str] = [s_for_split]
    for sep in seps:
        tmp: List[str] = []
        for p in parts:
            if sep in p:
                tmp.extend([x.strip() for x in p.split(sep) if x.strip()])
            else:
                tmp.append(p)
        parts = tmp

    tmp2: List[str] = []
    for p in parts:
        # Patch A/A2: if we treated a taxonomy label as atomic (pipe-encoded OR comma-encoded), NEVER split on commas.
        if (not _suppress_comma_split) and p.count(",") >= 2:
            tmp2.extend([x.strip() for x in p.split(",") if x.strip()])
        else:
            tmp2.append(p)

    out = [normalize_unicode_text(x) for x in tmp2 if normalize_unicode_text(x)]

    # Ensure recombined variants are included (and placed first) even if splitting already happened upstream.
    if recombined_variants:
        existing = {norm_title_key(x) for x in out}
        for rc in reversed(recombined_variants):
            k = norm_title_key(rc)
            if k and k not in existing:
                out.insert(0, rc)
                existing.add(k)

    return out



SIGNAL_MARKERS = re.compile(
    r"\b(department|dept\.?|school|college|division|faculty|center|centre|institute|program\b\s+in|committee|council|office|administration|administrative|unit)\b",
    re.I,
)


def has_signal_marker(s: str) -> bool:
    return bool(SIGNAL_MARKERS.search(normalize_unicode_text(s)))

# Structured signal encodings sometimes appear as prefix strings like:
#   department_unit:africana:crawl:Africana Studies
#   center_institute:race:crawl:Center for Race, Inequality...
# We want the human title portion to be matchable.
_STRUCTURED_SIGNAL_PREFIX = re.compile(
    r"^(department_unit|center_institute|committee_admin|other_nonprogram|junk|course|event_news|maybe_program|program)\s*:\s*[^:]*\s*:\s*(crawl|cv)\s*:\s*(.+)$",
    re.I,
)


def strip_structured_prefix(s: str) -> str:
    """Strip structured prefixes like `department_unit:bucket:src:Title` to just `Title`."""
    s0 = normalize_unicode_text(s)
    if not s0:
        return ""

    m = _STRUCTURED_SIGNAL_PREFIX.match(s0)
    if m:
        return normalize_unicode_text(m.group(3))

    # Fallback: if it looks like a 3+ segment colon prefix, take the last segment.
    # Only do this when the first segment is one of our known structured kinds.
    head = s0.split(":", 1)[0].strip().lower()
    if head in {
        "department_unit",
        "center_institute",
        "committee_admin",
        "other_nonprogram",
        "junk",
        "course",
        "event_news",
        "maybe_program",
        "program",
    } and s0.count(":") >= 3:
        return normalize_unicode_text(s0.split(":")[-1])

    return s0


def is_fragment_candidate(title: str) -> bool:
    t = normalize_unicode_text(title)
    if not t:
        return True

    # Patch B: clause fragments created by over-splitting often begin with conjunctions.
    # Examples: "and Gender Studies", "or Related Fields".
    if re.match(r"^(and|or|&)\b", t, flags=re.I):
        return True

    # Very short strings are almost always fragments from overly-split fields.
    if len(t) < 10:
        return True

    toks = _content_tokens(t)

    # One-token candidates (e.g., "Ethnic", "Gender") are usually fragment artifacts.
    if len(toks) < 2:
        return True

    return False


def signal_intent_bonus(ref: str, cand: str) -> float:
    """Boost when both ref and candidate look like signals; penalize mismatched intent."""
    r_sig = has_signal_marker(ref)
    c_sig = has_signal_marker(cand)
    if r_sig and c_sig:
        return 0.08
    if r_sig and not c_sig:
        return -0.05
    return 0.0


@dataclass(frozen=True)
class Candidate:
    title: str
    source: str  # crawl | cv | signal | unknown
    kind: str    # program | signal

    def title_key(self) -> str:
        return norm_title_key(self.title)


def _dedupe_preserve_order(items: Sequence[str]) -> List[str]:
    seen: Set[str] = set()
    out: List[str] = []
    for x in items:
        k = norm_title_key(x)
        if not k or k in seen:
            continue
        seen.add(k)
        out.append(x)
    return out


def parse_candidates_from_row(row: pd.Series) -> Tuple[str, List[Candidate], str]:
    """Return 2013 title and candidate list for a row, plus debug string."""
    t2013 = normalize_unicode_text(row.get("2013_program_name", ""))

    debug_added: List[str] = []

    # Crawl candidates
    crawl_raw = _split_program_name_field(row.get("program_titles_found", ""), debug_added=debug_added)

    # CV candidates (Patch 5: atomic_pipe_taxonomy protection)
    cv_raw = _split_program_name_field(
        row.get("college_vine_program_titles_found", ""),
        debug_added=debug_added,
        atomic_pipe_taxonomy=True
    )

    # Signals (optional)
    signal_raw: List[str] = []
    if "real_nonprogram_signals" in row.index:
        signal_raw = _split_program_name_field(row.get("real_nonprogram_signals", ""), debug_added=debug_added)

    # Second-pass recombine: if any candidate strings still contain '|', re-split and
    # inject recombined variants. This covers cases where upstream re-joined lists.
    def _second_pass_pipe_fix(items: List[str], atomic_pipe_taxonomy: bool = False) -> List[str]:
        fixed: List[str] = []
        for it in items:
            if "|" in normalize_unicode_text(it):
                fixed.extend(_split_program_name_field(
                    it,
                    debug_added=debug_added,
                    atomic_pipe_taxonomy=atomic_pipe_taxonomy,
                ))
            else:
                fixed.append(it)
        return fixed

    crawl_raw = _second_pass_pipe_fix(crawl_raw)
    cv_raw = _second_pass_pipe_fix(cv_raw, atomic_pipe_taxonomy=True)
    signal_raw = _second_pass_pipe_fix(signal_raw)

    candidates: List[Candidate] = []

    for t in crawl_raw:
        candidates.append(Candidate(title=t, source="crawl", kind="program"))

    for t in cv_raw:
        candidates.append(Candidate(title=t, source="cv", kind="program"))

    for t in signal_raw:
        t0 = strip_structured_prefix(t)
        src = "signal"
        # allow prefixed encodings like `crawl:Title` or `cv:Title`
        m = re.match(r"^(crawl|cv)\s*:\s*(.+)$", t0, flags=re.I)
        if m:
            src = "signal"
            t0 = normalize_unicode_text(m.group(2))
        t0 = strip_structured_prefix(t0)
        candidates.append(Candidate(title=t0, source=src, kind="signal"))

    # If the program pools contain explicit signal-marker titles, ALSO include them as signal-kind.
    # This helps match "X Department" even if it arrived via a noisy program_titles_found.
    for t in (crawl_raw + cv_raw):
        if has_signal_marker(t):
            candidates.append(Candidate(title=t, source="signal", kind="signal"))

    # Deduplicate by title key while preserving a preference for program-kind entries.
    # If both program and signal exist for the same title, keep both (for src=both accounting).
    # We'll handle src aggregation at match-time.
    debug_str = "||".join(_dedupe_preserve_order(debug_added))
    return t2013, candidates, debug_str


# ============================================================
# Synonym mapping: used ONLY for partial concordance + change tracking
# ============================================================

# Keep this intentionally conservative: we want to merge obvious equivalences without
# over-normalizing distinct programs.
# NOTE: applied ONLY in partial concordance scoring / alignment.
SYNONYM_REPLACEMENTS: List[Tuple[re.Pattern, str]] = [
    # ampersand handled earlier too, but keep for safety
    (re.compile(r"&", re.I), " and "),

    # common short forms
    (re.compile(r"\bdept\.?\b", re.I), " department "),
    (re.compile(r"\bprog\.?\b", re.I), " program "),

    # african-american variants
    (re.compile(r"\bafrican\s*[-–—]\s*american\b", re.I), " african american "),
    (re.compile(r"\bafro\s*[-–—]?\s*american\b", re.I), " african american "),

    # africology variants (seen in some catalogs)
    (re.compile(r"\bafricology\b", re.I), " africana studies "),

    # pan-african hyphenation / spacing variants
    (re.compile(r"\bpan\s*[-–—]?\s*african\b", re.I), " pan african "),
    (re.compile(r"\bpan\s+african\s+studies\b", re.I), " africana studies "),

    # black studies / africana common equivalences (very conservative)
    (re.compile(r"\bblack\s+studies\b", re.I), " africana studies "),

    # gender/women's apostrophe variants
    (re.compile(r"\bwomen['’]s\b", re.I), " women "),
    (re.compile(r"\bgender\s+and\s+sexuality\b", re.I), " gender sexuality "),

    # common misspelling seen in some sources
    (re.compile(r"\bcarribbean\b", re.I), " caribbean "),
]


def apply_synonym_map(s: str) -> str:
    """Apply conservative synonym replacements.

    This is ONLY used for partial concordance scoring / alignment and does not affect
    strict or loose exact concordance columns.
    """
    s = normalize_unicode_text(s)
    if not s:
        return ""
    out = s
    for rx, repl in SYNONYM_REPLACEMENTS:
        out = rx.sub(repl, out)
    out = normalize_unicode_text(out)
    return out


def _content_tokens(s: str) -> Set[str]:
    """Tokenize a title into content-bearing tokens for partial matching.

    IMPORTANT: synonym mapping is applied by the caller (partial-only). This function
    assumes it is receiving the already-preprocessed string.
    """
    s = norm_title_key_loose(s)
    if not s:
        return set()
    toks = [t for t in re.split(r"[\s_-]+", s) if t]
    toks = [t for t in toks if len(t) >= 3]  # remove tiny junk tokens

    # Drop common generic tail tokens that appear in fragmented pipe fields.
    drop = {"other", "misc", "general"}
    toks = [t for t in toks if t not in drop]

    return set(toks)


def _overlap_coeff(a: Set[str], b: Set[str]) -> float:
    """Token overlap similarity.

    Base is |A∩B| / min(|A|,|B|), but we guard against single-token candidates
    producing a perfect score against much longer references.
    """
    if not a or not b:
        return 0.0
    inter = len(a.intersection(b))
    la = len(a)
    lb = len(b)
    if inter == 0:
        return 0.0

    denom_min = float(min(la, lb))
    base = inter / denom_min if denom_min > 0 else 0.0

    # Guard: if one side is a single token but the other is meaningfully longer,
    # require broader coverage by down-weighting to an inter/max formulation.
    if min(la, lb) == 1 and max(la, lb) >= 3:
        denom_max = float(max(la, lb))
        return inter / denom_max if denom_max > 0 else 0.0

    return base


# ============================================================
# Domain-aware similarity helpers (used to boost best-available matches)
# ============================================================

# --- must-carry constraint for strong refs ---
# Accept hyphens/en-dashes/em-dashes between words (African-American, African–American, etc.)
_DASHSEP = r"[\s\-\u2010\u2011\u2012\u2013\u2014\u2212]+"
_AFRICAN_AMERICAN_RX = rf"\bafrican(?:{_DASHSEP})american\b"
_AFRO_AMERICAN_RX = rf"\bafro(?:{_DASHSEP})american\b"

_STRONG_REF_RX = re.compile(rf"\b(black|africana|{_AFRICAN_AMERICAN_RX}|{_AFRO_AMERICAN_RX})\b", re.I)
_STRONG_CAND_RX = re.compile(rf"\b(black|africana|{_AFRICAN_AMERICAN_RX}|{_AFRO_AMERICAN_RX})\b", re.I)

def is_strong_ref(ref: str) -> bool:
    """True if the reference expresses a strong Black/Africana/African-American intent.

    When True, candidates must "carry" at least one of {black, africana, african american}
    to be eligible for fuzzy/backstop matching (see `candidate_eligible_under_strong_ref`).
    """
    r = normalize_unicode_text(ref)
    return bool(_STRONG_REF_RX.search(r))

def candidate_has_strong_token(cand: str) -> bool:
    c = normalize_unicode_text(cand)
    return bool(_STRONG_CAND_RX.search(c))

def candidate_eligible_under_strong_ref(ref: str, cand: str, *, allow_black_credential_exception: bool) -> bool:
    """Eligibility gating to prevent african-only candidates from matching africana/black refs.

    Rules:
      - If ref is strong (black/africana/african-american), candidate must contain at least one
        of {black, africana, african american} to be eligible for ANY fuzzy match OR backstop.
      - Exception (only when `allow_black_credential_exception=True`): allow candidates that
        are credentialed AND contain black (e.g., "Black American Studies Minor").
      - If ref is not strong (pure african, ethnic, race), no extra gating is applied.
    """
    if not is_strong_ref(ref):
        return True

    # Normal path: must carry a strong token.
    if candidate_has_strong_token(cand):
        return True

    if allow_black_credential_exception:
        c = normalize_unicode_text(cand)
        if _CREDENTIAL_MARKERS.search(c) and re.search(r"\bblack\b", c, flags=re.I):
            return True

    return False

DOMAIN_FAMILY_PATTERNS: Dict[str, List[re.Pattern]] = {
    # NOTE: keep conservative; these are only used as a small bonus and for backstop gating
    "black": [
        re.compile(r"\bblack\b", re.I),
        re.compile(_AFRICAN_AMERICAN_RX, re.I),
        re.compile(_AFRO_AMERICAN_RX, re.I),
        re.compile(r"\bblack\s+diaspor", re.I),
        re.compile(r"\bpan[-\s]?african\b", re.I),
    ],
    "africana": [
        re.compile(r"\bafricana\b", re.I),
        re.compile(r"\bpan[-\s]?african\b", re.I),
        re.compile(r"\bafrican\s+diaspor", re.I),
    ],
    "african": [
        re.compile(r"\bafrican\b", re.I),
        re.compile(r"\bafrica\b", re.I),
    ],
    "ethnic": [
        re.compile(r"\bethnic\b", re.I),
        re.compile(r"\bethnicity\b", re.I),
        re.compile(r"\bmulticultural(?:ism)?\b", re.I),
        re.compile(r"\bmulti[-\s]?cultural(?:ism)?\b", re.I),
        re.compile(r"\bmulti[-\s]?ethnic\b", re.I),
        re.compile(r"\binter[-\s]?cultural\b", re.I),
        re.compile(r"\bcultural\s+minority\b", re.I),
        re.compile(r"\bmigration\b", re.I),
        re.compile(r"\bchicanx\b", re.I),
        re.compile(r"\blatinx\b", re.I),
        re.compile(r"\basian\s+american\b", re.I),
        re.compile(r"\bnative\s+american\b", re.I),
    ],
    "race": [
        re.compile(r"\brace\b", re.I),
        re.compile(r"\bracial\b", re.I),
        re.compile(r"\bjustice\b", re.I),
        re.compile(r"\bequity\b", re.I),
    ],
}

# Related-family mapping for best-available fallback matches
RELATED_DOMAIN_FAMILIES: Dict[str, Set[str]] = {
    "black": {"black", "africana", "african"},
    "africana": {"africana", "black", "african"},
    "african": {"african", "africana", "black"},
    "ethnic": {"ethnic", "race"},
    "race": {"race", "ethnic"},
}

# ============================================================
# CV taxonomy label heuristic (optional rejection)
# ============================================================

# Very common CollegeVine category labels that are not full program entities (normalized keys).
_KNOWN_CV_TAXONOMY_LABELS: Set[str] = {
    norm_title_key(x)
    for x in [
        "Ethnic Studies",
        "Race and Ethnicity",
        "African American Studies",
        "Africana Studies",
        "African Studies",
        "Black Studies",
        "Gender Studies",
        "Women's Studies",
        "Women and Gender Studies",
        "Latinx Studies",
        "Chicanx Studies",
        "Asian American Studies",
        "Native American Studies",
        "Indigenous Studies",
        "Queer Studies",
        "LGBTQ Studies",
    ]
}

def is_cv_taxonomy_label(title: str) -> bool:
    """Heuristic: True if a CV title looks like a taxonomy/category label rather than a program entity.
    Apply only when the candidate source is CollegeVine (source == 'cv').
    """
    t = normalize_unicode_text(title)
    if not t:
        return False

    # If it's explicitly credentialed or signal-like, treat as a real entity.
    if has_credential_marker(t) or has_signal_marker(t):
        return False

    k = norm_title_key(t)
    if k in _KNOWN_CV_TAXONOMY_LABELS:
        return True

    # Generic short labels (<=3 tokens) ending in "studies" are often taxonomy-like.
    toks = [x for x in norm_title_key(t).split() if x]
    if len(toks) <= 3 and toks and toks[-1] == "studies":
        return True

    return False

# ============================================================
# Winner-selection (Patch 1): prefer best on-site primary academic titles
# ============================================================

_PRIMARY_ACAD_RX = re.compile(
    r"\b(program|major|minor|department|dept\.?|studies)\b",
    re.I,
)

_SECONDARY_ORG_RX = re.compile(
    r"\b(center|centre|institute|committee|council|association)\b",
    re.I,
)

# ============================================================
# Patch 4: Rename-family rescue (Africana / African-American / Black Studies)
# ============================================================

# Trigger when the 2013 reference indicates a strong Africana/Black/African-American intent.
_RENAME_FAMILY_REF_RX = re.compile(
    rf"\b(africana|black|pan{_DASHSEP}?african|{_AFRICAN_AMERICAN_RX}|{_AFRO_AMERICAN_RX})\b|\bafrican\s+and\s+african\s+american\b",
    re.I,
)


def is_rename_family_ref(ref: str) -> bool:
    r = normalize_unicode_text(ref)
    return bool(r and _RENAME_FAMILY_REF_RX.search(r))


def rename_family_rescue_match(
    ref: str,
    candidates: List[Candidate],
    kind_pref: str,
    threshold: float = 0.55,
    *,
    allow_category_mapping: bool = False,
) -> Tuple[bool, str, float, str]:
    """Rename-family rescue tier.

    Purpose: rescue common institutional retitles/renames among:
      Africana ↔ African-American ↔ Black Studies ↔ Pan-African

    Gating (low risk):
      - ref must be in rename-family intent (`is_rename_family_ref`)
      - candidate must be primary-academic (Patch 1 classifier), OR (taxonomy only when no on-site primary exists)
      - candidate must be in the related rename family (black/africana/african)
      - apply standard nav/year/mega/nonprogram filters

    Special case: allow African Studies (african-only) ONLY as a last resort when no
    stronger black/africana candidates exist for the row.
    """
    ref0 = normalize_unicode_text(ref)
    if not ref0:
        return False, "", 0.0, ""

    if not is_rename_family_ref(ref0):
        return False, "", 0.0, ""

    # Candidate pool by kind
    pool_cands: List[Candidate] = []
    seen: Set[str] = set()
    for c in candidates:
        if c.kind != kind_pref:
            continue
        k = norm_title_key(c.title)
        if not k or k in seen:
            continue
        seen.add(k)
        pool_cands.append(c)

    # Standard filters
    pool_cands = [c for c in pool_cands if not is_nav_prefix_title(c.title)]
    pool_cands = [c for c in pool_cands if not is_mega_string_candidate(c.title)]

    if not has_year_token(ref0):
        pool_cands = [c for c in pool_cands if not has_year_token(c.title)]

    if kind_pref == "program":
        pool_cands = [c for c in pool_cands if not is_nonprogram_title(c.title)]
    else:
        if not has_signal_marker(ref0):
            pool_cands = [c for c in pool_cands if not is_nonprogram_title(c.title)]
        else:
            pool_cands = [c for c in pool_cands if (not is_nonprogram_title(c.title)) or has_signal_marker(c.title)]

    if not pool_cands:
        return False, "", 0.0, ""

    # Pre-compute whether any strong (black/africana/african-american) candidate exists.
    def _sources_set_for_title(title: str) -> Set[str]:
        k = norm_title_key(title)
        return {c.source for c in candidates if norm_title_key(c.title) == k}

    strong_exists = False
    onsite_primary_exists = False
    for c in pool_cands:
        t = normalize_unicode_text(c.title)
        if not t:
            continue
        srcs = _sources_set_for_title(t)
        cls = candidate_class(t, srcs)
        fams = domain_families_present(t)
        if cls == "primary_academic" and srcs != {"cv"}:
            onsite_primary_exists = True
        if ("black" in fams) or ("africana" in fams) or candidate_has_strong_token(t):
            strong_exists = True

    best_score = 0.0
    best_title = ""
    best_detail = ""

    f_ref = domain_families_present(ref0)

    for c in pool_cands:
        cand0 = normalize_unicode_text(c.title)
        if not cand0:
            continue

        # Must be non-fragment
        cand_tok = _content_tokens(cand0)
        if len(cand_tok) < 2 or is_fragment_candidate(cand0):
            continue

        srcs = _sources_set_for_title(cand0)
        cls = candidate_class(cand0, srcs)

        # Require primary academic; allow taxonomy only when explicitly allowed OR
        # when we have no on-site primary academic survivors.
        if cls == "taxonomy":
            if (not allow_category_mapping) and onsite_primary_exists:
                continue
        elif cls != "primary_academic":
            continue

        f_cand = domain_families_present(cand0)
        if not f_cand:
            continue

        # Must be in the rename family neighborhood.
        if not (f_cand.intersection({"black", "africana", "african"})):
            continue

        # Special case: African-only candidates are allowed only as a last resort.
        african_only = ("african" in f_cand) and ("black" not in f_cand) and ("africana" not in f_cand) and (not candidate_has_strong_token(cand0))
        if african_only and strong_exists:
            continue

        # Scoring (synonyms + parenthetical drop) using shared scorer.
        sc, a_best, b_best = best_partial_title_match(
            [ref0],
            [cand0],
            use_synonyms=True,
            drop_parenthetical=True,
        )

        # Apply additional domain + intent adjustments to the score so it behaves like other tiers.
        # (best_partial_title_match already includes domain_bonus/signal_intent/entity_type_penalty)
        sc = float(sc)

        if sc > best_score:
            best_score = sc
            best_title = b_best or cand0
            best_detail = (
                f'rename_family_rescue: ref="{ref0}" ~ cand="{best_title}" score={best_score:.2f} '
                f'fam_ref={sorted(f_ref)} fam_cand={sorted(f_cand)} cls={cls} african_only={int(african_only)}'
            )

    if best_title and best_score >= float(threshold):
        return True, best_title, float(best_score), best_detail

    return False, "", float(best_score), ""

# ============================================================
# Patch 2: Entity-type penalty for center/institute drift
# ============================================================

# Apply a soft penalty when the reference is a primary academic unit (dept/program/studies)
# but the candidate is an organizational entity (center/institute/committee/etc.).
# This improves specificity while retaining sensitivity (the candidate can still win
# if no better academic unit exists).
_ENTITY_TYPE_PENALTY = 0.15


def _looks_primary_academic_ref(ref: str) -> bool:
    r = normalize_unicode_text(ref)
    if not r:
        return False
    if has_credential_marker(r):
        return True
    if _PRIMARY_ACAD_RX.search(r):
        return True
    # short noun phrase ending in Studies
    toks = [x for x in norm_title_key(r).split() if x]
    if toks and toks[-1] == "studies":
        return True
    return False


def _looks_secondary_org_candidate(cand: str) -> bool:
    c = normalize_unicode_text(cand)
    if not c:
        return False
    return bool(_SECONDARY_ORG_RX.search(c))


def entity_type_penalty(ref: str, cand: str) -> float:
    """Return a negative penalty when a secondary org (center/institute/etc.) is matching
    a primary academic reference (dept/program/studies).

    Penalty is soft (subtracts ~0.15) and is applied in similarity scoring.
    """
    if _looks_primary_academic_ref(ref) and _looks_secondary_org_candidate(cand):
        return -_ENTITY_TYPE_PENALTY
    return 0.0


def candidate_class(cand_title: str, cand_sources: Set[str]) -> str:
    """Classify a candidate title for winner-selection.

    Returns one of: primary_academic | secondary_org | taxonomy | other

    - taxonomy is reserved for CV taxonomy/category labels when the title exists ONLY via cv.
    - primary_academic includes program-ish entities (Program/Major/Minor/Department/Studies).
    """
    t = normalize_unicode_text(cand_title)
    if not t:
        return "other"

    # Taxonomy: only when the title is *only* from CV and looks like a CV category label.
    only_cv = (cand_sources == {"cv"})
    if only_cv and is_cv_taxonomy_label(t):
        return "taxonomy"

    if _SECONDARY_ORG_RX.search(t):
        return "secondary_org"

    if _PRIMARY_ACAD_RX.search(t) or has_credential_marker(t):
        return "primary_academic"

    # Short noun-phrase program names (e.g., "Africana Studies")
    toks = [x for x in norm_title_key(t).split() if x]
    if 1 < len(toks) <= 3 and toks[-1] == "studies":
        return "primary_academic"

    return "other"


def _score_pair_for_winner(ref: str, cand: str) -> Tuple[float, float, bool]:
    """Score a single (ref, cand) pair for winner-selection.

    Returns: (score_0_to_1, ref_coverage_0_to_1, ref_subset_of_cand_tokens)

    Uses the same scoring primitives as fuzzy matching: max(token overlap, ref coverage, seq ratio)
    plus domain + intent bonuses, and applies nonprogram penalties.
    """
    r0 = normalize_unicode_text(ref)
    c0 = normalize_unicode_text(cand)
    if not r0 or not c0:
        return 0.0, 0.0, False

    # Reject/penalize obvious non-program navigation/marketing titles.
    reject_np, np_pen = nonprogram_penalty(r0, c0)
    if reject_np:
        return 0.0, 0.0, False

    r_can = canonicalize_program_title(r0, drop_parenthetical=True)
    c_can = canonicalize_program_title(c0, drop_parenthetical=True)

    # Use conservative synonym mapping for winner-selection consistency
    r_pre = apply_synonym_map(r_can)
    c_pre = apply_synonym_map(c_can)

    r_loose = norm_title_key_loose(r_pre)
    c_loose = norm_title_key_loose(c_pre)

    r_tok = _content_tokens(r_pre)
    c_tok = _content_tokens(c_pre)

    tok_score = _overlap_coeff(r_tok, c_tok)
    coverage = (len(r_tok.intersection(c_tok)) / float(len(r_tok))) if r_tok else 0.0
    seq_score = difflib.SequenceMatcher(None, r_loose, c_loose).ratio() if (r_loose and c_loose) else 0.0

    base = max(tok_score, coverage, seq_score)

    if np_pen > 0:
        base = max(0.0, base * (1.0 - np_pen))
    # Patch 6: soft demotion for boilerplate titles that still contain academic keywords.
    bp_pen = boilerplate_penalty_fraction(c0)
    if bp_pen > 0:
        base = max(0.0, base * (1.0 - bp_pen))
    # Fragment / short-candidate penalty (same spirit as best_partial_title_match)
    short_penalty = 0.0
    if len(c_tok) < 2:
        short_penalty += 0.25
    if len(c0) < 14:
        short_penalty += 0.10
    if len(r_tok) >= 3 and len(c_tok) == 1:
        short_penalty += 0.20
    if short_penalty > 0:
        base = max(0.0, base * (1.0 - min(0.60, short_penalty)))

    bonus = domain_bonus(r0, c0, r_tok, c_tok) + signal_intent_bonus(r0, c0) + entity_type_penalty(r0, c0)
    score = min(1.0, float(base + bonus))

    subset = bool(r_tok) and r_tok.issubset(c_tok)
    return score, float(coverage), subset


def select_best_title_patch1(
    ref: str,
    candidates: List[Candidate],
    *,
    primary_thresh: float = 0.80,
) -> Tuple[str, float, str, str]:
    """Select the best title using Patch 1 precedence.

    Returns (best_title, best_score, best_source, winner_reason).

    Precedence:
      1) primary_academic with score >= primary_thresh (prefer on-site if available)
      2) secondary_org (prefer on-site if available)
      3) taxonomy backstop

    Tie-breakers:
      - Higher score
      - Prefer ref token subset of candidate tokens
      - Higher ref coverage
      - Shorter/cleaner strings when scores are close
    """
    ref0 = normalize_unicode_text(ref)
    if not ref0:
        return "", 0.0, "", ""

    # Build unique titles -> sources + kind presence
    title_to_sources: Dict[str, Set[str]] = {}
    for c in candidates:
        t = normalize_unicode_text(c.title)
        if not t:
            continue
        k = norm_title_key(t)
        if not k:
            continue
        title_to_sources.setdefault(k, set()).add(c.source)

    # Candidate pool: apply the same high-level rejects used in matching.
    unique_titles: List[str] = []
    seen_k: Set[str] = set()
    for c in candidates:
        t = normalize_unicode_text(c.title)
        if not t:
            continue
        k = norm_title_key(t)
        if not k or k in seen_k:
            continue
        seen_k.add(k)

        # Apply absolute rejects
        if is_nav_prefix_title(t):
            continue
        if is_mega_string_candidate(t):
            continue
        if (not has_year_token(ref0)) and has_year_token(t):
            continue

        # Must-carry constraint: never allow african-only candidates to match strong refs.
        if not candidate_eligible_under_strong_ref(ref0, t, allow_black_credential_exception=False):
            continue

        # Patch B2: hard fragment filter for Patch1 winner selection.
        # Prevent titles like "and Gender Studies" from winning under taxonomy_backstop.
        if is_fragment_candidate(t):
            continue

        unique_titles.append(t)

    if not unique_titles:
        return "", 0.0, "", ""

    # Score all candidates
    scored: List[Tuple[str, float, float, bool, str]] = []
    # tuple: (title, score, coverage, subset, class)
    for t in unique_titles:
        sc, cov, subset = _score_pair_for_winner(ref0, t)
        srcs = title_to_sources.get(norm_title_key(t), set())
        cls = candidate_class(t, srcs)
        scored.append((t, float(sc), float(cov), bool(subset), cls))

    # Helper: prefer on-site sources when available
    def _is_onsite(srcs: Set[str]) -> bool:
        return bool(srcs.intersection({"crawl", "signal", "both"}))

    def _pick_best(pool: List[Tuple[str, float, float, bool, str]], reason: str, prefer_onsite: bool) -> Tuple[str, float, str, str]:
        if not pool:
            return "", 0.0, "", ""

        # If requested, restrict to on-site if any exist.
        if prefer_onsite:
            onsite_pool = []
            for t, sc, cov, subset, cls in pool:
                srcs = title_to_sources.get(norm_title_key(t), set())
                if _is_onsite(srcs):
                    onsite_pool.append((t, sc, cov, subset, cls))
            if onsite_pool:
                pool = onsite_pool

        # Sort by: score desc, subset desc, coverage desc, length asc
        pool_sorted = sorted(
            pool,
            key=lambda x: (-x[1], -int(x[3]), -x[2], len(normalize_unicode_text(x[0]))),
        )

        # When scores are very close, prefer shorter/cleaner strings
        best = pool_sorted[0]
        best_title, best_score, best_cov, best_subset, best_cls = best

        best_src = _sources_for_title(candidates, best_title)
        return best_title, float(best_score), best_src, reason

    # 1) primary academic above threshold
    prim = [(t, sc, cov, subset, cls) for (t, sc, cov, subset, cls) in scored if cls == "primary_academic" and sc >= float(primary_thresh)]
    bt, bs, bsrc, breason = _pick_best(prim, "patch1:primary_academic", prefer_onsite=True)
    if bt:
        return bt, bs, bsrc, breason

    # 2) secondary org (no threshold beyond >0)
    sec = [(t, sc, cov, subset, cls) for (t, sc, cov, subset, cls) in scored if cls == "secondary_org" and sc > 0.0]
    bt, bs, bsrc, breason = _pick_best(sec, "patch1:secondary_org", prefer_onsite=True)
    if bt:
        return bt, bs, bsrc, breason

    # 3) taxonomy backstop
    tax = [(t, sc, cov, subset, cls) for (t, sc, cov, subset, cls) in scored if cls == "taxonomy" and sc > 0.0]
    bt, bs, bsrc, breason = _pick_best(tax, "patch1:taxonomy_backstop", prefer_onsite=False)
    if bt:
        return bt, bs, bsrc, breason

    # If nothing classified, fall back to best-scoring non-fragment title
    other = [(t, sc, cov, subset, cls) for (t, sc, cov, subset, cls) in scored if sc > 0.0 and not is_fragment_candidate(t)]
    bt, bs, bsrc, breason = _pick_best(other, "patch1:best_available", prefer_onsite=True)
    return bt, bs, bsrc, breason

# ============================================================
# Family-only rescue stage (gated, low-lexical-overlap)
# ============================================================

# Anchor keywords required for the rescue gate. This prevents overly-broad family matches.
_FAMILY_RESCUE_ANCHOR_RX = re.compile(
    r"\b("
    r"race|racial|ethnic|ethnicity|diaspora|"
    r"african|africana|black|"
    r"latinx|indigenous|sovereignty|"
    r"queer|gender|women"
    r")\b",
    re.I,
)

# Families that are considered strong enough to trigger rescue when present in the ref.
_FAMILY_RESCUE_STRONG_FAMS: Set[str] = {"ethnic", "race", "black", "africana"}


def _family_rescue_anchor_hit(title: str) -> str:
    """Return the matching anchor keyword (lowercased) if present, else empty."""
    t = normalize_unicode_text(title)
    m = _FAMILY_RESCUE_ANCHOR_RX.search(t) if t else None
    return (m.group(1).lower() if m else "")

def family_only_rescue_match(
    ref: str,
    candidates: List[Candidate],
    kind_pref: str,
    threshold: float = 0.52,
    *,
    allow_category_mapping: bool = False,
) -> Tuple[bool, str, float, str]:
    """Family-only rescue stage.

    Purpose: allow a lower similarity threshold when *direct* domain-family overlap is strong,
    but lexical overlap is otherwise weak.

    Gating:
      - Ref must contain at least one "strong" family (ethnic/race/black/africana)
      - Candidate must share a DIRECT overlapping family with the ref (not merely related)
      - Candidate must contain at least one rescue anchor keyword
      - Candidate must be non-fragment (>=2 content tokens)
      - Candidate must pass standard nonprogram/nav/year/mega-string filters

    Additional guards:
      - Structural mismatch: reject program ↔ credentialed item (minor/certificate/etc.) unless both sides agree.
      - CV taxonomy/category labels are rejected by default (unless allow_category_mapping=True).
    """
    ref0 = normalize_unicode_text(ref)
    if not ref0:
        return False, "", 0.0, ""

    f_ref = domain_families_present(ref0)
    if not f_ref or not (f_ref.intersection(_FAMILY_RESCUE_STRONG_FAMS)):
        return False, "", 0.0, ""

    ref_has_cred = has_credential_marker(ref0)

    # Work with Candidate objects so we can apply CV taxonomy heuristics using source.
    pool_cands: List[Candidate] = []
    seen: Set[str] = set()
    for c in candidates:
        if c.kind != kind_pref:
            continue
        k = norm_title_key(c.title)
        if not k or k in seen:
            continue
        seen.add(k)
        pool_cands.append(c)

    # Standard filters (mirror fuzzy/related tiers)
    pool_cands = [c for c in pool_cands if not is_nav_prefix_title(c.title)]
    pool_cands = [c for c in pool_cands if not is_mega_string_candidate(c.title)]

    if not has_year_token(ref0):
        pool_cands = [c for c in pool_cands if not has_year_token(c.title)]

    # Anti-nonprogram eligibility filtering (same policy as other tiers)
    if kind_pref == "program":
        pool_cands = [c for c in pool_cands if not is_nonprogram_title(c.title)]
    else:
        if not has_signal_marker(ref0):
            pool_cands = [c for c in pool_cands if not is_nonprogram_title(c.title)]
        else:
            pool_cands = [c for c in pool_cands if (not is_nonprogram_title(c.title)) or has_signal_marker(c.title)]

    if not pool_cands:
        return False, "", 0.0, ""

    best_score = 0.0
    best_title = ""
    best_detail = ""

    for c in pool_cands:
        cand0 = normalize_unicode_text(c.title)
        if not cand0:
            continue

        # Structural mismatch guard: program ↔ credentialed item (minor/cert/etc.)
        cand_has_cred = has_credential_marker(cand0)
        if ref_has_cred != cand_has_cred:
            continue

        # Default: reject CV taxonomy/category labels unless explicitly allowed.
        if (not allow_category_mapping) and c.source == "cv" and is_cv_taxonomy_label(cand0):
            continue

        # Must-carry constraint: protect strong black/africana refs
        if not candidate_eligible_under_strong_ref(ref0, cand0, allow_black_credential_exception=False):
            continue

        # Fragment suppression
        cand_tok = _content_tokens(cand0)
        if len(cand_tok) < 2 or is_fragment_candidate(cand0):
            continue

        # Direct family overlap only (not just related)
        f_cand = domain_families_present(cand0)
        if not f_cand:
            continue
        fam_overlap = f_ref.intersection(f_cand)
        if not fam_overlap:
            continue

        # Anchor keyword requirement
        anchor = _family_rescue_anchor_hit(cand0)
        if not anchor:
            continue

        sc, a_best, b_best = best_partial_title_match(
            [ref0],
            [cand0],
            use_synonyms=True,
            drop_parenthetical=True,
        )

        if sc > best_score:
            best_score = float(sc)
            best_title = b_best or cand0
            best_detail = (
                f'family_rescue: ref="{ref0}" ~ cand="{best_title}" score={best_score:.2f} '
                f'anchor={anchor} fam_ref={sorted(f_ref)} fam_cand={sorted(f_cand)} fam_overlap={sorted(fam_overlap)}'
            )

    if best_title and best_score >= float(threshold):
        return True, best_title, best_score, best_detail

    return False, "", best_score, ""


# Normalize dash variants for family extraction (domain_families_present operates on a
# lightly normalized string, not the loose key). This ensures patterns like
# `multi[-\s]?ethnic` match “Multi‑Ethnic” / “Multi–Ethnic” / etc.
_DASH_VARIANTS = re.compile(r"[\u2010\u2011\u2012\u2013\u2014\u2015\u2212]")


def _preprocess_for_family_extraction(s: str) -> str:
    s0 = normalize_unicode_text(s)
    if not s0:
        return ""
    # Standardize unicode dash variants to ASCII hyphen so family regexes can match reliably.
    s0 = _DASH_VARIANTS.sub("-", s0)
    s0 = _WS_RE.sub(" ", s0).strip()
    return s0


def domain_families_present(s: str) -> Set[str]:
    s0 = _preprocess_for_family_extraction(s)
    if not s0:
        return set()
    fams: Set[str] = set()
    for fam, pats in DOMAIN_FAMILY_PATTERNS.items():
        for rx in pats:
            if rx.search(s0):
                fams.add(fam)
                break
    return fams


def domain_related(f1: Set[str], f2: Set[str]) -> bool:
    if not f1 or not f2:
        return False
    for a in f1:
        rel = RELATED_DOMAIN_FAMILIES.get(a, {a})
        if rel.intersection(f2):
            return True
    return False


def domain_bonus(ref: str, cand: str, ref_tok: Set[str], cand_tok: Set[str]) -> float:
    """Small bonus for strong domain-keyword alignment.

    Applies only when both sides have at least 2 content tokens to avoid rewarding
    fragment candidates (e.g., "Ethnic").
    """
    if len(ref_tok) < 2 or len(cand_tok) < 2:
        return 0.0

    f_ref = domain_families_present(ref)
    f_cand = domain_families_present(cand)
    if not f_ref or not f_cand:
        return 0.0

    # Asymmetry penalty: ref is black/africana but candidate is african-only.
    # This prevents "African Studies" from scoring like a near-match to "Africana/Black" programs.
    if ("black" in f_ref or "africana" in f_ref) and ("african" in f_cand) and ("black" not in f_cand) and ("africana" not in f_cand):
        return -0.10

    # Strong bonus when families overlap directly
    if f_ref.intersection(f_cand):
        return 0.12

    # Smaller bonus for related families (e.g., africana ~ african)
    if domain_related(f_ref, f_cand):
        return 0.06

    return 0.0


def best_partial_title_match(
    a_titles: List[str],
    b_titles: List[str],
    use_synonyms: bool = False,
    drop_parenthetical: bool = False,
) -> Tuple[float, str, str]:
    """Return best partial match score and the (a_title, b_title) pair.

    Score is max(token overlap coefficient, SequenceMatcher ratio) on loose-normalized strings.

    If `use_synonyms` is True, apply `apply_synonym_map()` BEFORE loose normalization.
    If `drop_parenthetical` is True, drop parenthetical segments before scoring.

    This is used ONLY for partial concordance + change tracking.
    """
    best_score = 0.0
    best_a = ""
    best_b = ""

    for a in (a_titles or []):
        a_raw = normalize_unicode_text(a)
        if not a_raw:
            continue
        a_can = canonicalize_program_title(a_raw, drop_parenthetical=drop_parenthetical)
        a_pre = apply_synonym_map(a_can) if use_synonyms else a_can
        a_loose = norm_title_key_loose(a_pre)
        a_tok = _content_tokens(a_pre)

        for b in (b_titles or []):
            b_raw = normalize_unicode_text(b)
            if not b_raw:
                continue
            # Reject/penalize obvious non-program navigation/marketing titles.
            reject_np, np_pen = nonprogram_penalty(a_raw, b_raw)
            if reject_np:
                continue
            b_can = canonicalize_program_title(b_raw, drop_parenthetical=drop_parenthetical)
            b_pre = apply_synonym_map(b_can) if use_synonyms else b_can
            b_loose = norm_title_key_loose(b_pre)
            b_tok = _content_tokens(b_pre)

            tok_score = _overlap_coeff(a_tok, b_tok)

            # Ref-coverage: how much of the reference token set is explained by the candidate
            coverage = (len(a_tok.intersection(b_tok)) / float(len(a_tok))) if a_tok else 0.0

            seq_score = (
                difflib.SequenceMatcher(None, a_loose, b_loose).ratio() if (a_loose and b_loose) else 0.0
            )

            base = max(tok_score, coverage, seq_score)

            # Apply soft non-program penalties (multiplicative)
            if np_pen > 0:
                base = max(0.0, base * (1.0 - np_pen))

            # Patch 6: soft demotion for boilerplate titles that still contain academic keywords.
            bp_pen = boilerplate_penalty_fraction(b_raw)
            if bp_pen > 0:
                base = max(0.0, base * (1.0 - bp_pen))

            # (1) Fragment / short-candidate penalty: discourage single-word or very short candidates
            # from winning as best matches (common artifact from over-splitting).
            short_penalty = 0.0
            if len(b_tok) < 2:
                short_penalty += 0.25
            if len(b_raw) < 14:
                short_penalty += 0.10
            if len(a_tok) >= 3 and len(b_tok) == 1:
                short_penalty += 0.20

            # Apply penalties multiplicatively to preserve ordering among strong candidates.
            if short_penalty > 0:
                base = max(0.0, base * (1.0 - min(0.60, short_penalty)))

            # Domain + intent bonuses (small, capped) + Patch 2 entity-type penalty
            bonus = domain_bonus(a_raw, b_raw, a_tok, b_tok)
            bonus += signal_intent_bonus(a_raw, b_raw)
            bonus += entity_type_penalty(a_raw, b_raw)

            score = min(1.0, float(base + bonus))

            if score > best_score:
                best_score = score
                best_a = a_raw
                best_b = b_raw

    return best_score, best_a, best_b

# ============================================================
# Patch 6: Boilerplate / navigational-title demotion
# ============================================================

# Phrases commonly emitted by hub pages that are not real program titles.
_BOILERPLATE_RX = re.compile(
    r"\b("
    r"check\s+out|prospective\s+students\s*:|welcome\s+to|"
    r"read\s+more|learn\s+more|please\s+visit|click\s+here|"
    r"find\s+out\s+more|explore\s+the"
    r")\b",
    re.I,
)

_BOILERPLATE_SAFE_KEYWORDS = re.compile(
    r"\b(studies|department|dept\.?|program|major|minor|concentration|certificate)\b",
    re.I,
)


def boilerplate_drop(title: str) -> bool:
    """Return True if `title` looks like boilerplate AND lacks strong academic keywords.

    Patch 6 rule: drop boilerplate outright unless it contains a strong keyword like 'Studies',
    'Department', 'Program', etc. If it contains a strong keyword, keep it but apply a penalty
    in scoring (see `boilerplate_penalty_fraction`).
    """
    t = normalize_unicode_text(title)
    if not t:
        return True
    if not _BOILERPLATE_RX.search(t):
        return False
    # Drop if it does not contain any strong academic keyword.
    return not bool(_BOILERPLATE_SAFE_KEYWORDS.search(t))


def boilerplate_penalty_fraction(title: str) -> float:
    """Return a soft penalty fraction for boilerplate titles that we still keep.

    Patch 6 rule: if boilerplate AND it *does* contain a strong academic keyword, demote it.
    """
    t = normalize_unicode_text(title)
    if not t:
        return 1.0
    if _BOILERPLATE_RX.search(t) and _BOILERPLATE_SAFE_KEYWORDS.search(t):
        return 0.25
    return 0.0

#
# Heuristic: titles that are likely marketing/navigation pages rather than program entities.
# For PROGRAM-kind pools, these should be rejected outright.
# For SIGNAL-kind pools, they are only allowed when BOTH ref and candidate are signal-like.
_NONPROGRAM_PREFIX = re.compile(
    r"^(why|about|welcome|visit|check\s+out|read\s+more|learn\s+more|prospective\s+students)\b",
    re.I,
)
_NONPROGRAM_CONTAINS = re.compile(
    r"\b("
    r"news|event|events|donate|faculty|staff|"
    r"calendar|meeting|committee\s+meeting|committee|"
    r"updated|please\s+visit|note\s*:|"
    r"curriculum\s+review"
    r")\b",
    re.I,
)

def is_nav_prefix_title(title: str) -> bool:
    """True if title starts with navigation/marketing prefixes (absolute reject in all pools)."""
    t = normalize_unicode_text(title)
    return bool(t and _NONPROGRAM_PREFIX.search(t))



def is_nonprogram_title(title: str) -> bool:
    """Return True if `title` looks like navigation/marketing/non-program content."""
    t = normalize_unicode_text(title)
    if not t:
        return True
    if is_nav_prefix_title(t):
        return True
    if _NONPROGRAM_CONTAINS.search(t):
        return True
    if t.endswith("?"):
        return True
    return False

# Year-like strings (e.g., 2025-26, 2025–2026, 2025/26, 2025)
_YEARISH = re.compile(r"\b(19\d{2}|20\d{2})(?:\s*[-/\u2010-\u2015]\s*\d{2,4})?\b")

def has_year_token(s: str) -> bool:
    """True if the string contains a year or year-range like 2025-26."""
    t = normalize_unicode_text(s)
    return bool(t and _YEARISH.search(t))


# ============================================================
# Mega-string reject heuristic
# ============================================================

# Candidates that are "everything on the page" (mega-strings) can score ~1.0 by accident.
# We reject them early to protect specificity.
_MEGA_NAV_PHRASES = re.compile(
    r"\b(welcome\s+to|prospective\s+students|check\s+out|read\s+more|learn\s+more|visit)\b",
    re.I,
)

# Count credential markers (allow repeats). This is intentionally broader than _CREDENTIAL_MARKERS.
_MEGA_CREDENTIAL_MARKERS = re.compile(
    r"\b(ba|b\.?a\.?|bs|b\.?s\.?|ma|m\.?a\.?|ms|m\.?s\.?|phd|ph\.?d\.?|bachelor|master|doctoral|major|minor|certificate|concentration|degree\s+type)\b",
    re.I,
)


def is_mega_string_candidate(title: str, *, token_threshold: int = 22) -> bool:
    """Return True if `title` looks like an over-concatenated mega-string.

    Heuristic:
      - Many tokens (>= token_threshold)
      - AND (multiple credential markers OR obvious nav phrases)
      - OR extremely long token counts (>= 30) regardless
    """
    t = normalize_unicode_text(title)
    if not t:
        return True

    toks = [x for x in norm_title_key(t).split() if x]
    n_tok = len(toks)

    if n_tok >= 30:
        return True

    if n_tok < token_threshold:
        return False

    cred_count = len(_MEGA_CREDENTIAL_MARKERS.findall(t))
    has_nav = bool(_MEGA_NAV_PHRASES.search(t))

    # Require at least some additional evidence beyond just being long.
    if has_nav:
        return True
    if cred_count >= 2:
        return True

    return False


def nonprogram_penalty(ref: str, cand: str) -> Tuple[bool, float]:
    """Return (reject, penalty_fraction).

    This function is used inside similarity scoring loops.
    """
    r_sig = has_signal_marker(ref)
    c_sig = has_signal_marker(cand)

    t = normalize_unicode_text(cand)
    if not t:
        return True, 1.0

    # Absolute reject for nav/marketing prefix titles in ALL pools.
    if is_nav_prefix_title(t):
        return True, 1.0

    # For other nonprogram (news/events/donate/faculty/etc.), only allow when BOTH ref and cand are signal-like.
    if is_nonprogram_title(t):
        if r_sig and c_sig:
            return False, 0.0
        return True, 1.0

    # Patch 6: demote (or drop) boilerplate titles.
    # Drop is handled by upstream filters where possible; here we apply a soft penalty for
    # boilerplate strings that still contain strong academic keywords.
    bp_pen = boilerplate_penalty_fraction(t)
    if bp_pen > 0:
        return False, bp_pen

    return False, 0.0

# ============================================================
# Match ladder
# ============================================================


def _sources_for_title(cands: List[Candidate], title: str) -> str:
    """Return crawl/cv/signal/both/unknown summary for a title."""
    k = norm_title_key(title)
    srcs = sorted({c.source for c in cands if norm_title_key(c.title) == k})

    # normalize crawl/cv present alongside signal
    has_crawl = "crawl" in srcs
    has_cv = "cv" in srcs

    if has_crawl and has_cv:
        return "both"
    if has_crawl:
        return "crawl"
    if has_cv:
        return "cv"
    if "signal" in srcs:
        return "signal"
    if srcs:
        return srcs[0]
    return "unknown"


def _kinds_for_title(cands: List[Candidate], title: str) -> Set[str]:
    k = norm_title_key(title)
    return {c.kind for c in cands if norm_title_key(c.title) == k}


def _pick_best_of_kind(
    ref: str,
    candidates: List[Candidate],
    kind_pref: str,
    level: str,
    detail_label: str,
    threshold: float,
    use_synonyms: bool,
    canonical: bool,
    allow_category_mapping: bool = False,
) -> Tuple[bool, str, float, str]:
    """Try to match within candidates restricted by kind.

    Returns (ok, best_title, score, detail). Always returns diagnostic `detail` even when no match is returned.

    Fuzzy scoring uses max(token overlap, ref-coverage, SequenceMatcher) plus a small domain-keyword bonus (capped at 1.0).
    """
    ref = normalize_unicode_text(ref)
    if not ref:
        return False, "", 0.0, ""

    def _diag_score_pair(r: str, c: str) -> float:
        """Diagnostic-only similarity score used to explain filtered/suppressed candidates.

        IMPORTANT: This MUST NOT affect match outcomes. It intentionally ignores
        nonprogram/mega/year rejection logic and fragment suppression.
        """
        r0 = normalize_unicode_text(r)
        c0 = normalize_unicode_text(c)
        if not r0 or not c0:
            return 0.0

        drop_paren = True if canonical else False

        r_can = canonicalize_program_title(r0, drop_parenthetical=drop_paren)
        c_can = canonicalize_program_title(c0, drop_parenthetical=drop_paren)

        r_pre = apply_synonym_map(r_can) if use_synonyms else r_can
        c_pre = apply_synonym_map(c_can) if use_synonyms else c_can

        r_loose = norm_title_key_loose(r_pre)
        c_loose = norm_title_key_loose(c_pre)

        r_tok = _content_tokens(r_pre)
        c_tok = _content_tokens(c_pre)

        tok_score = _overlap_coeff(r_tok, c_tok)
        coverage = (len(r_tok.intersection(c_tok)) / float(len(r_tok))) if r_tok else 0.0
        seq_score = difflib.SequenceMatcher(None, r_loose, c_loose).ratio() if (r_loose and c_loose) else 0.0

        base = max(tok_score, coverage, seq_score)
        bonus = domain_bonus(r0, c0, r_tok, c_tok) + signal_intent_bonus(r0, c0) + entity_type_penalty(r0, c0)
        return float(min(1.0, max(0.0, base + bonus)))

    raw_pool = [c.title for c in candidates if c.kind == kind_pref]
    raw_pool = _dedupe_preserve_order(raw_pool)

    def _title_sources(title: str) -> Set[str]:
        k = norm_title_key(title)
        return {c.source for c in candidates if norm_title_key(c.title) == k}

    removed: Dict[str, List[str]] = {}

    def _apply_filter(name: str, items: List[str], keep_fn) -> List[str]:
        kept: List[str] = []
        dropped: List[str] = []
        for it in items:
            if keep_fn(it):
                kept.append(it)
            else:
                dropped.append(it)
        if dropped:
            removed.setdefault(name, []).extend(dropped)
        return kept

    pool = list(raw_pool)

    # Absolute rejects (always removed)
    pool = _apply_filter("nav_prefix", pool, lambda t: not is_nav_prefix_title(t))
    pool = _apply_filter("mega_string", pool, lambda t: not is_mega_string_candidate(t))
    pool = _apply_filter("boilerplate_drop", pool, lambda t: not boilerplate_drop(t))
    # Reject year-tagged titles unless the reference itself includes a year.
    if not has_year_token(ref):
        pool = _apply_filter("year_token", pool, lambda t: not has_year_token(t))

    # Nonprogram filtering policy:
    # - Program-kind pools: reject any nonprogram title outright.
    # - Signal-kind pools: allow nonprogram ONLY when BOTH ref and candidate are signal-like.
    if kind_pref == "program":
        pool = _apply_filter("nonprogram", pool, lambda t: not is_nonprogram_title(t))
    else:
        if not has_signal_marker(ref):
            pool = _apply_filter("nonprogram", pool, lambda t: not is_nonprogram_title(t))
        else:
            pool = _apply_filter("nonprogram", pool, lambda t: (not is_nonprogram_title(t)) or has_signal_marker(t))

    # Patch 3: CV taxonomy labels are backstop-only unless explicitly allowed.
    # If any on-site candidates (crawl/signal) survive filtering, drop CV taxonomy labels.
    if (not allow_category_mapping) and kind_pref == "program":
        try:
            has_onsite_survivor = any(
                ("cv" not in _title_sources(t)) or (len(_title_sources(t)) > 1)
                for t in pool
            )
        except Exception:
            has_onsite_survivor = False

        if has_onsite_survivor:
            pool = _apply_filter(
                "cv_taxonomy_backstop",
                pool,
                lambda t: not ("cv" in _title_sources(t) and _title_sources(t) == {"cv"} and is_cv_taxonomy_label(t)),
            )

    # Must-carry constraint: never allow african-only candidates to match strong refs (black/africana/african-american)
    # for ANY fuzzy mode.
    if level not in {"strict_raw", "strict_canonical"}:
        pool = _apply_filter(
            "strong_ref_gate",
            pool,
            lambda t: candidate_eligible_under_strong_ref(ref, t, allow_black_credential_exception=False),
        )

    if not pool:
        # Diagnostic-only: report the best-looking excluded candidate and why it was excluded.
        best_reason = ""
        best_cand = ""
        best_sc = 0.0
        for reason, items in removed.items():
            for it in items:
                sc = _diag_score_pair(ref, it)
                if sc > best_sc:
                    best_sc = sc
                    best_cand = it
                    best_reason = reason

        if best_cand:
            return (
                False,
                "",
                float(best_sc),
                f'{detail_label}: rejected by {best_reason} filter cand="{best_cand}" score={best_sc:.2f}',
            )

        return False, "", 0.0, ""

    # STRICT levels
    if level == "strict_raw":
        ref_k = norm_title_key(ref)
        keys = {norm_title_key(t): t for t in pool}
        if ref_k in keys:
            t = keys[ref_k]
            return True, t, 1.0, f'{detail_label}: ref="{ref}" == cand="{t}"'
        return False, "", 0.0, ""

    if level == "strict_canonical":
        ref_can = canonicalize_program_title(ref, drop_parenthetical=True)
        ref_k = norm_title_key(ref_can)
        keys = {norm_title_key(canonicalize_program_title(t, drop_parenthetical=True)): t for t in pool}
        if ref_k in keys:
            t = keys[ref_k]
            return True, t, 1.0, f'{detail_label}: ref="{ref}" == cand="{t}"'
        return False, "", 0.0, ""

    # FUZZY levels
    drop_paren = True if canonical else False
    # Hard gate: if the candidate pool contains only fragment-like titles, we still score,
    # but we will not allow a fragment to be returned as the best match.
    fragment_keys = {norm_title_key(t) for t in pool if is_fragment_candidate(t)}

    score, a_best, b_best = best_partial_title_match(
        [ref],
        pool,
        use_synonyms=use_synonyms,
        drop_parenthetical=drop_paren,
    )

    if b_best:
        # Do not allow fragments to win as best matches.
        if norm_title_key(b_best) in fragment_keys:
            return (
                False,
                "",
                float(score),
                f'{detail_label}: suppressed fragment candidate cand="{b_best}" score={score:.2f}',
            )

        if score >= float(threshold):
            detail = f'{detail_label}: ref="{a_best}" ~ cand="{b_best}" score={score:.2f}'
            return True, b_best, float(score), detail

        # Diagnostic-only: below threshold but still report best pair.
        return (
            False,
            "",
            float(score),
            f'{detail_label}: below_threshold ref="{a_best}" ~ cand="{b_best}" score={score:.2f}',
        )

    return False, "", float(score), ""




def related_domain_backstop_match(
    ref: str,
    candidates: List[Candidate],
    kind_pref: str,
    threshold: float = 0.62,
    *,
    allow_category_mapping: bool = False,
) -> Tuple[bool, str, float, str]:
    """Best-available related match used only when all other modes fail.

    Gated by domain-family presence and requires non-fragment candidates.
    CV taxonomy/category labels are rejected by default unless allow_category_mapping=True, and credential mismatch is blocked.
    """
    ref0 = normalize_unicode_text(ref)
    if not ref0:
        return False, "", 0.0, ""

    f_ref = domain_families_present(ref0)
    if not f_ref:
        return False, "", 0.0, ""

    ref_has_cred = has_credential_marker(ref0)

    # Candidate pool by kind preference (preserve Candidate objects so we can apply CV taxonomy heuristics)
    pool_cands: List[Candidate] = []
    seen: Set[str] = set()
    for c in candidates:
        if c.kind != kind_pref:
            continue
        k = norm_title_key(c.title)
        if not k or k in seen:
            continue
        seen.add(k)
        pool_cands.append(c)

    # Standard filters
    pool_cands = [c for c in pool_cands if not is_nav_prefix_title(c.title)]
    pool_cands = [c for c in pool_cands if not is_mega_string_candidate(c.title)]

    # Reject year-tagged titles unless the reference itself includes a year.
    if not has_year_token(ref0):
        pool_cands = [c for c in pool_cands if not has_year_token(c.title)]

    # Anti-nonprogram eligibility filtering (same policy as fuzzy pools)
    if kind_pref == "program":
        pool_cands = [c for c in pool_cands if not is_nonprogram_title(c.title)]
    else:
        if not has_signal_marker(ref0):
            pool_cands = [c for c in pool_cands if not is_nonprogram_title(c.title)]
        else:
            pool_cands = [c for c in pool_cands if (not is_nonprogram_title(c.title)) or has_signal_marker(c.title)]

    if not pool_cands:
        return False, "", 0.0, ""

    best_score = 0.0
    best_title = ""
    best_detail = ""

    # Use the same partial matcher scoring (now includes coverage + domain bonus)
    for c in pool_cands:
        cand0 = normalize_unicode_text(c.title)
        if not cand0:
            continue
        # Structural mismatch guard: block program ↔ credentialed item (minor/cert/etc.) unless both sides agree.
        cand_has_cred = has_credential_marker(cand0)
        if ref_has_cred != cand_has_cred:
            continue

        # Default: reject CV taxonomy/category labels unless explicitly allowed.
        if (not allow_category_mapping) and c.source == "cv" and is_cv_taxonomy_label(cand0):
            continue

        # Must-carry constraint: strong refs cannot backstop-match to african-only candidates.
        if not candidate_eligible_under_strong_ref(ref0, cand0, allow_black_credential_exception=False):
            continue

        # Fragment suppression: require >=2 content tokens on candidate
        ref_tok = _content_tokens(ref0)
        cand_tok = _content_tokens(cand0)
        if len(cand_tok) < 2:
            continue

        # If the 2013 ref is signal-like, require the candidate to also look signal-like
        # for the signal-kind backstop. This prevents program titles from stealing signal intent.
        if has_signal_marker(ref0) and kind_pref == "signal" and not has_signal_marker(cand0):
            continue

        f_cand = domain_families_present(cand0)
        if not f_cand:
            continue

        # Must be same-family or related-family
        if not (f_ref.intersection(f_cand) or domain_related(f_ref, f_cand)):
            continue

        sc, a_best, b_best = best_partial_title_match(
            [ref0],
            [cand0],
            use_synonyms=True,
            drop_parenthetical=True,
        )

        if sc > best_score:
            best_score = float(sc)
            best_title = b_best or cand0
            best_detail = (
                f'related_domain_backstop: ref="{ref0}" ~ cand="{best_title}" score={best_score:.2f} '
                f'fam_ref={sorted(f_ref)} fam_cand={sorted(f_cand)}'
            )

    if best_title and best_score >= float(threshold):
        return True, best_title, best_score, best_detail

    return False, "", best_score, ""

# --- related_credential tier ---
_CREDENTIAL_MARKERS = re.compile(r"\b(minor|major|certificate|concentration)\b", re.I)

def has_credential_marker(s: str) -> bool:
    """True if the title explicitly encodes a credential type (minor/major/certificate/concentration)."""
    t = normalize_unicode_text(s)
    return bool(t and _CREDENTIAL_MARKERS.search(t))

def related_credential_tier_match(
    ref: str,
    candidates: List[Candidate],
    kind_pref: str,
    threshold: float,
) -> Tuple[bool, str, float, str]:
    """Related-but-not-same credential tier.

    Fires only when:
      - ref is in a domain family (black/africana/african/ethnic/race)
      - candidate contains an in-family keyword (domain family present)
      - candidate contains a credential marker (minor/major/certificate/concentration)
      - candidate is not a fragment (>=2 content tokens)
      - score clears a slightly-lower-than-fuzzy threshold
    """
    ref0 = normalize_unicode_text(ref)
    if not ref0:
        return False, "", 0.0, ""

    f_ref = domain_families_present(ref0)
    if not f_ref:
        return False, "", 0.0, ""

    pool = _dedupe_preserve_order([c.title for c in candidates if c.kind == kind_pref])
    pool = [t for t in pool if not is_nav_prefix_title(t)]
    pool = [t for t in pool if not is_mega_string_candidate(t)]

    # Reject year-tagged titles unless the reference itself includes a year.
    if not has_year_token(ref0):
        pool = [t for t in pool if not has_year_token(t)]
    # Anti-nonprogram eligibility filtering (related_credential is still a program-tier)
    if kind_pref == "program":
        pool = [t for t in pool if not is_nonprogram_title(t)]
    else:
        if not has_signal_marker(ref0):
            pool = [t for t in pool if not is_nonprogram_title(t)]
        else:
            pool = [t for t in pool if (not is_nonprogram_title(t)) or has_signal_marker(t)]
    if not pool:
        return False, "", 0.0, ""

    best_score = 0.0
    best_title = ""
    best_detail = ""

    ref_tok = _content_tokens(ref0)
    if len(ref_tok) < 2:
        return False, "", 0.0, ""

    for cand in pool:
        cand0 = normalize_unicode_text(cand)
        if not cand0:
            continue
        # Must-carry constraint with a narrow exception: allow only credentialed candidates that contain "black"
        # (e.g., "Black American Studies Minor"), but never plain "African Studies".
        if not candidate_eligible_under_strong_ref(ref0, cand0, allow_black_credential_exception=True):
            continue

        # Must look like a credentialed program item
        if not _CREDENTIAL_MARKERS.search(cand0):
            continue

        cand_tok = _content_tokens(cand0)
        if len(cand_tok) < 2:
            continue

        f_cand = domain_families_present(cand0)
        if not f_cand:
            continue

        # Require same-family or related-family alignment
        if not (f_ref.intersection(f_cand) or domain_related(f_ref, f_cand)):
            continue

        sc, a_best, b_best = best_partial_title_match(
            [ref0],
            [cand0],
            use_synonyms=True,
            drop_parenthetical=True,
        )

        if sc > best_score:
            best_score = float(sc)
            best_title = b_best or cand0
            best_detail = (
                f'related_credential: ref="{ref0}" ~ cand="{best_title}" score={best_score:.2f} '
                f'fam_ref={sorted(f_ref)} fam_cand={sorted(f_cand)}'
            )

    if best_title and best_score >= float(threshold):
        return True, best_title, best_score, best_detail

    return False, "", best_score, ""


def match_2013_to_candidates(
    t2013: str,
    candidates: List[Candidate],
    fuzzy_threshold: float = 0.80,
    *,
    allow_category_mapping: bool = False,
) -> Tuple[str, str, str, str, float, str]:
    """Return best match tuple:

      (best_title, best_source, best_kind, match_level, match_score, detail)
    """
    t2013 = normalize_unicode_text(t2013)
    if not t2013:
        return "", "", "", "NO_MATCH", 0.0, ""

    is_sig_2013 = has_signal_marker(t2013)

    # primary kind preference based on the 2013 string
    # If 2013 is signal-like, we heavily prefer signal candidates.
    primary_kind = "signal" if is_sig_2013 else "program"
    secondary_kind = "program" if primary_kind == "signal" else "signal"

    def _resolve_best_kind(chosen_kind_pref: str, kinds: Set[str]) -> str:
        """Resolve output kind for a matched title.

        - If the chosen ladder step was signal-preferred, and the title exists as a signal candidate, report signal.
        - Else, if the title exists only as signal, report signal.
        - Otherwise report program.
        """
        if chosen_kind_pref == "signal" and "signal" in kinds:
            return "signal"
        if "signal" in kinds and "program" not in kinds:
            return "signal"
        return "program"

    ladder = [
        ("strict_raw", primary_kind, 1.0, False, False),
        ("strict_raw", secondary_kind, 1.0, False, False),
        ("strict_canonical", primary_kind, 1.0, False, True),
        ("strict_canonical", secondary_kind, 1.0, False, True),
        ("fuzzy_raw", primary_kind, fuzzy_threshold, False, False),
        ("fuzzy_raw", secondary_kind, fuzzy_threshold, False, False),
        ("fuzzy_canonical", primary_kind, fuzzy_threshold, False, True),
        ("fuzzy_canonical", secondary_kind, fuzzy_threshold, False, True),
        ("fuzzy_syn", primary_kind, fuzzy_threshold, True, True),
        ("fuzzy_syn", secondary_kind, fuzzy_threshold, True, True),
    ]

    best_diag_detail = ""
    best_diag_score = 0.0

    chosen_best: Optional[Tuple[str, str, str, str, float, str]] = None
    for level, kind, thr, use_syn, use_can in ladder:
        ok, best_title, score, detail = _pick_best_of_kind(
            t2013,
            candidates,
            kind_pref=kind,
            level="strict_raw" if level == "strict_raw" else ("strict_canonical" if level == "strict_canonical" else "fuzzy"),
            detail_label=level,
            threshold=thr,
            use_synonyms=use_syn,
            canonical=use_can,
            allow_category_mapping=allow_category_mapping,
        )

        if (not ok) and detail and float(score) >= float(best_diag_score):
            best_diag_score = float(score)
            best_diag_detail = detail

        # The helper uses "fuzzy" for fuzzy modes; map to ladder label for output.
        if ok and best_title:
            best_source = _sources_for_title(candidates, best_title)
            kinds = _kinds_for_title(candidates, best_title)
            best_kind = _resolve_best_kind(kind, kinds)

            out_level = level
            out_score = 1.0 if level.startswith("strict") else float(score)
            out_detail = detail if detail else f"{level}: matched"
            chosen_best = (best_title, best_source, best_kind, out_level, out_score, out_detail)
            break

    # Tier: Patch 4 rename-family rescue (Africana / African-American / Black Studies)
    # Only fires when ref indicates rename-family intent, and only considers primary-academic candidates.
    ok, best_title, score, detail = rename_family_rescue_match(
        t2013,
        candidates,
        kind_pref=primary_kind,
        threshold=0.55,
        allow_category_mapping=allow_category_mapping,
    )
    if not ok:
        ok, best_title, score, detail = rename_family_rescue_match(
            t2013,
            candidates,
            kind_pref=secondary_kind,
            threshold=0.55,
            allow_category_mapping=allow_category_mapping,
        )

    if ok and best_title:
        best_source = _sources_for_title(candidates, best_title)
        kinds = _kinds_for_title(candidates, best_title)
        best_kind = _resolve_best_kind(primary_kind, kinds)
        chosen_best = (best_title, best_source, best_kind, "rename_family_rescue", float(score), detail)

    # Tier: related-but-not-same credential match (only after all strict/fuzzy modes fail)
    f_ref = domain_families_present(t2013)
    if ("ethnic" in f_ref) or ("race" in f_ref):
        related_credential_threshold = max(0.66, min(0.76, float(fuzzy_threshold) - 0.08))
    else:
        related_credential_threshold = max(0.70, min(0.78, float(fuzzy_threshold) - 0.06))

    ok, best_title, score, detail = related_credential_tier_match(
        t2013,
        candidates,
        kind_pref=primary_kind,
        threshold=related_credential_threshold,
    )
    if not ok:
        ok, best_title, score, detail = related_credential_tier_match(
            t2013,
            candidates,
            kind_pref=secondary_kind,
            threshold=related_credential_threshold,
        )

    if ok and best_title:
        best_source = _sources_for_title(candidates, best_title)
        kinds = _kinds_for_title(candidates, best_title)
        best_kind = _resolve_best_kind(primary_kind, kinds)
        chosen_best = (best_title, best_source, best_kind, "related_credential", float(score), detail)

    # Tier: family-only rescue (gated) — sits between related_credential and related_domain_backstop
    # Lower threshold is allowed only when direct family overlap + anchor constraints pass.
    if ("ethnic" in f_ref) or ("race" in f_ref):
        rescue_threshold = 0.50
    else:
        rescue_threshold = 0.52

    ok, best_title, score, detail = family_only_rescue_match(
        t2013,
        candidates,
        kind_pref=primary_kind,
        threshold=rescue_threshold,
        allow_category_mapping=allow_category_mapping,
    )
    if not ok:
        ok, best_title, score, detail = family_only_rescue_match(
            t2013,
            candidates,
            kind_pref=secondary_kind,
            threshold=rescue_threshold,
            allow_category_mapping=allow_category_mapping,
        )

    if ok and best_title:
        best_source = _sources_for_title(candidates, best_title)
        kinds = _kinds_for_title(candidates, best_title)
        best_kind = _resolve_best_kind(primary_kind, kinds)
        chosen_best = (best_title, best_source, best_kind, "family_rescue", float(score), detail)

    # Backstop: best-available related-domain match (only when nothing else matched)
    if ("ethnic" in f_ref) or ("race" in f_ref):
        related_threshold = max(0.50, min(0.58, float(fuzzy_threshold) - 0.22))
    else:
        related_threshold = max(0.50, min(0.62, float(fuzzy_threshold) - 0.15))

    # Prefer the same kind ordering used above
    ok, best_title, score, detail = related_domain_backstop_match(
        t2013,
        candidates,
        kind_pref=primary_kind,
        threshold=related_threshold,
        allow_category_mapping=allow_category_mapping,
    )
    if not ok:
        ok, best_title, score, detail = related_domain_backstop_match(
            t2013,
            candidates,
            kind_pref=secondary_kind,
            threshold=related_threshold,
            allow_category_mapping=allow_category_mapping,
        )

    if ok and best_title:
        best_source = _sources_for_title(candidates, best_title)
        kinds = _kinds_for_title(candidates, best_title)
        best_kind = _resolve_best_kind(primary_kind, kinds)
        chosen_best = (best_title, best_source, best_kind, "related_domain_backstop", float(score), detail)

    # ------------------------------------------------------------
    # Patch 1: final winner-selection override
    # ------------------------------------------------------------
    # If we already have a selected match, we may override it to prefer the best
    # on-site primary academic title over secondary org titles and CV taxonomy labels.
    patch1_title, patch1_score, patch1_source, patch1_reason = select_best_title_patch1(
        t2013,
        candidates,
        primary_thresh=float(fuzzy_threshold),
    )

    if patch1_title:
        # If we had no match at all, adopt Patch 1 pick.
        if chosen_best is None:
            kinds = _kinds_for_title(candidates, patch1_title)
            best_kind = "signal" if ("signal" in kinds and "program" not in kinds) else "program"
            return patch1_title, patch1_source, best_kind, "winner_override", float(patch1_score), (
                f"winner_override: {patch1_reason} | ref=\"{normalize_unicode_text(t2013)}\" ~ cand=\"{patch1_title}\" score={patch1_score:.2f}"
            )

        # Otherwise, override only when Patch 1 yields a different, higher-precedence candidate
        # and is not meaningfully worse in score.
        cur_title, cur_source, cur_kind, cur_level, cur_score, cur_detail = chosen_best

        if norm_title_key(patch1_title) != norm_title_key(cur_title):
            # Allow a small score delta when upgrading precedence (e.g., department beats center).
            if float(patch1_score) >= float(cur_score) - 0.05:
                kinds = _kinds_for_title(candidates, patch1_title)
                best_kind = "signal" if ("signal" in kinds and "program" not in kinds) else "program"
                return patch1_title, patch1_source, best_kind, "winner_override", float(patch1_score), (
                    f"winner_override: replaced=\"{cur_title}\" ({cur_level}) -> \"{patch1_title}\"; {patch1_reason} | score={patch1_score:.2f}"
                )

        # If Patch 1 agrees with the chosen title, return the chosen result.
        return chosen_best

    # No Patch 1 candidate; return chosen_best if present.
    if chosen_best is not None:
        return chosen_best

    if chosen_best is not None:
        return chosen_best
    return "", "", "", "NO_MATCH", 0.0, (best_diag_detail or "")


def any_match_under_any_mode(
    t2013: str,
    candidates: List[Candidate],
    fuzzy_threshold: float = 0.80,
) -> Set[str]:
    """Return set of title keys that match 2013 under any mode.

    Used to compute discovered__new_titles_unmatched.
    """
    t2013 = normalize_unicode_text(t2013)
    if not t2013:
        return set()

    matched: Set[str] = set()
    pools = {
        "program": _dedupe_preserve_order([c.title for c in candidates if c.kind == "program"]),
        "signal": _dedupe_preserve_order([c.title for c in candidates if c.kind == "signal"]),
    }
    pools["program"] = [t for t in pools["program"] if not is_mega_string_candidate(t)]
    pools["signal"] = [t for t in pools["signal"] if not is_mega_string_candidate(t)]

    # Reject year-tagged titles unless the reference itself includes a year.
    if not has_year_token(t2013):
        pools["program"] = [t for t in pools["program"] if not has_year_token(t)]
        pools["signal"] = [t for t in pools["signal"] if not has_year_token(t)]

    # Apply the same anti-nonprogram eligibility filtering used by the match ladder.
    pools["program"] = [t for t in pools["program"] if not is_nonprogram_title(t)]

    if not has_signal_marker(t2013):
        pools["signal"] = [t for t in pools["signal"] if not is_nonprogram_title(t)]
    else:
        pools["signal"] = [
            t
            for t in pools["signal"]
            if (not is_nonprogram_title(t)) or has_signal_marker(t)
        ]

    # strict raw
    ref_k = norm_title_key(t2013)
    for kind, pool in pools.items():
        for t in pool:
            if kind == "signal" and is_nonprogram_title(t):
                if not (has_signal_marker(t2013) and has_signal_marker(t)):
                    continue
            if norm_title_key(t) == ref_k:
                matched.add(norm_title_key(t))

    # strict canonical
    ref_can = canonicalize_program_title(t2013, drop_parenthetical=True)
    ref_ck = norm_title_key(ref_can)
    for kind, pool in pools.items():
        for t in pool:
            if kind == "signal" and is_nonprogram_title(t):
                if not (has_signal_marker(t2013) and has_signal_marker(t)):
                    continue
            if norm_title_key(canonicalize_program_title(t, drop_parenthetical=True)) == ref_ck:
                matched.add(norm_title_key(t))

    # fuzzy raw
    for kind, pool in pools.items():
        sc, a_best, b_best = best_partial_title_match([t2013], pool, use_synonyms=False, drop_parenthetical=False)
        if sc >= float(fuzzy_threshold) and b_best:
            matched.add(norm_title_key(b_best))

    # fuzzy canonical
    for kind, pool in pools.items():
        sc, a_best, b_best = best_partial_title_match([t2013], pool, use_synonyms=False, drop_parenthetical=True)
        if sc >= float(fuzzy_threshold) and b_best:
            matched.add(norm_title_key(b_best))

    # fuzzy synonym
    for kind, pool in pools.items():
        sc, a_best, b_best = best_partial_title_match([t2013], pool, use_synonyms=True, drop_parenthetical=True)
        if sc >= float(fuzzy_threshold) and b_best:
            matched.add(norm_title_key(b_best))

    return matched


def _pipe_join(items: Sequence[str]) -> str:
    items = [normalize_unicode_text(x) for x in items if normalize_unicode_text(x)]
    return "|".join(items)


def build_output_for_row(
    row: pd.Series,
    fuzzy_threshold: float,
    allow_category_mapping: bool = False,
) -> Dict[str, object]:
    t2013, candidates, debug_recombined = parse_candidates_from_row(row)

    best_title, best_source, best_kind, match_level, match_score, detail = match_2013_to_candidates(
        t2013,
        candidates,
        fuzzy_threshold=fuzzy_threshold,
        allow_category_mapping=bool(allow_category_mapping),
    )

    is_sig_2013 = int(has_signal_marker(t2013))

    # Candidate summaries
    crawl_titles = _dedupe_preserve_order([c.title for c in candidates if c.source == "crawl" and c.kind == "program"])
    cv_titles = _dedupe_preserve_order([c.title for c in candidates if c.source == "cv" and c.kind == "program"])
    signal_titles = _dedupe_preserve_order([c.title for c in candidates if c.kind == "signal"])

    # display lists in canonical display form (keeping parentheticals)
    crawl_disp = [canonicalize_program_title(t) for t in crawl_titles]
    cv_disp = [canonicalize_program_title(t) for t in cv_titles]
    sig_disp = [canonicalize_program_title(t) for t in signal_titles]

    all_titles = _dedupe_preserve_order(crawl_titles + cv_titles + signal_titles)

    matched_keys_any = any_match_under_any_mode(t2013, candidates, fuzzy_threshold=fuzzy_threshold)
    new_unmatched = [t for t in all_titles if norm_title_key(t) not in matched_keys_any]

    new_program_when_best_signal: List[str] = []
    if best_kind == "signal":
        # show program-ish discovered titles besides the best match
        prog_all = _dedupe_preserve_order(crawl_titles + cv_titles)
        new_program_when_best_signal = [t for t in prog_all if norm_title_key(t) != norm_title_key(best_title)]

    return {
        "match_2013__best_title": best_title,
        "match_2013__best_source": best_source,
        "match_2013__best_kind": best_kind,
        "match_2013__match_level": match_level,
        "match_2013__match_score": float(match_score),
        "match_2013__detail": detail,
        "match_2013__is_signal_marker_in_2013": is_sig_2013,
        "debug__recombined_candidates_added": debug_recombined,
        "discovered__program_titles__crawl": _pipe_join(crawl_disp),
        "discovered__program_titles__cv": _pipe_join(cv_disp),
        "discovered__signal_titles": _pipe_join(sig_disp),
        "discovered__all_titles": _pipe_join(all_titles),
        "discovered__new_titles_unmatched": _pipe_join(new_unmatched),
        "discovered__new_program_titles_when_best_signal": _pipe_join(new_program_when_best_signal),
    }


#
# ============================================================
# Unit tests (family extraction ordering / dash handling)
# ============================================================

class TestDomainFamilyExtraction(unittest.TestCase):
    def test_ethnic_family_multicultural_gender_program(self):
        s = "Multicultural and Gender Studies Program"
        fams = domain_families_present(s)
        self.assertIn("ethnic", fams)

    def test_ethnic_family_multi_ethnic_program(self):
        # Use a dash that is common in scraped text (unicode hyphen or en dash)
        # to ensure our family extraction preprocessing is robust.
        for s in [
            "Multi-Ethnic Studies Program",          # ASCII hyphen
            "Multi‑Ethnic Studies Program",      # non-breaking hyphen
            "Multi–Ethnic Studies Program",      # en dash
            "Multi−Ethnic Studies Program",      # minus sign
        ]:
            fams = domain_families_present(s)
            self.assertIn("ethnic", fams, msg=f"failed for: {s!r}")


def _run_tests() -> int:
    suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestDomainFamilyExtraction)
    runner = unittest.TextTestRunner(verbosity=2)
    result = runner.run(suite)
    return 0 if result.wasSuccessful() else 1


def derive_output_path(input_path: str) -> str:
    base, ext = os.path.splitext(input_path)
    if not ext:
        ext = ".csv"
    return f"{base}__2013_current_matches{ext}"


def main(argv: Optional[Sequence[str]] = None) -> int:
    ap = argparse.ArgumentParser(
        description="Match 2013 program name to current discovered titles within unitid",
        allow_abbrev=False,
    )
    ap.add_argument(
        "--input",
        default="ace_unitid_merge__ace_x_2013comp__webscrape__v15simple__bucketed_programs.csv",
        help="Input CSV (wide table)",
    )
    ap.add_argument(
        "--output",
        default="",
        help="Output CSV (default: input with __2013_current_matches suffix)",
    )
    ap.add_argument(
        "--fuzzy-threshold",
        type=float,
        default=0.80,
        help="Threshold for fuzzy matching (default: 0.80)",
    )
    ap.add_argument(
        "--allow-category-mapping",
        action="store_true",
        help="Allow CV taxonomy/category labels (e.g., 'Ethnic Studies') to be eligible in rescue/backstop tiers.",
    )
    ap.add_argument(
        "--run-tests",
        action="store_true",
        help="Run unit tests for family extraction / normalization and exit",
    )

    # In notebooks (ipykernel), sys.argv often includes `-f <kernel.json>`.
    # With `allow_abbrev=True` (argparse default), `-f` can be mis-read as an
    # abbreviation for `--fuzzy-threshold`. We disable abbreviation and ignore
    # unknown args when argv is None (i.e., using sys.argv).
    if argv is None:
        args, unknown = ap.parse_known_args()
        # If you run this as a script and pass unknown flags, surface them.
        # In IPython/Jupyter, `-f` is expected and should be ignored.
        if unknown:
            # ipykernel adds: ['-f', '/path/to/kernel.json']
            if not (len(unknown) == 2 and unknown[0] == "-f" and unknown[1].endswith(".json")):
                print(f"WARNING: ignoring unknown args: {unknown}", file=sys.stderr)
    else:
        args = ap.parse_args(argv)

    if getattr(args, "run_tests", False):
        return _run_tests()

    in_path = args.input
    out_path = args.output or derive_output_path(in_path)

    if not os.path.exists(in_path):
        print(f"ERROR: input not found: {in_path}", file=sys.stderr)
        return 2

    df = pd.read_csv(in_path, dtype=str, keep_default_na=False)

    # basic required columns
    required = ["unitid", "2013_program_name", "program_titles_found", "college_vine_program_titles_found"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        print(f"ERROR: missing required columns: {missing}", file=sys.stderr)
        return 2

    out_rows: List[Dict[str, object]] = []
    for _, row in df.iterrows():
        out_rows.append(
            build_output_for_row(
                row,
                fuzzy_threshold=float(args.fuzzy_threshold),
                allow_category_mapping=bool(getattr(args, "allow_category_mapping", False)),
            )
        )

    out_df = df.copy()
    for k in out_rows[0].keys() if out_rows else []:
        out_df[k] = [r.get(k, "") for r in out_rows]

    out_df.to_csv(out_path, index=False, quoting=csv.QUOTE_MINIMAL)
    print(f"Wrote: {out_path}")
    return 0


if __name__ == "__main__":
    #raise SystemExit(main())
    main()

Wrote: ace_unitid_merge__ace_x_2013comp__webscrape__v15simple__bucketed_programs__2013_current_matches.csv


In [38]:
# ============================================================
# PATCH README 
# ============================================================
"""
Simplified Output Splitter for 2013 vs Current Program Matching
==============================================================

Purpose
-------
This notebook cell reads the master CSV produced by the 2013-vs-current matching pipeline and
writes several simplified CSVs that are easier to review by humans. The master file remains the
source of truth for detailed debugging and analysis; these outputs are intentionally column-reduced
and re-ordered so that "2013 program" -> "current program" comparison is immediate.

Files Produced
--------------
All outputs are written to OUT_DIR.

01_inadequate_controls.csv
    Rows where we did NOT have adequate control counts from either crawl-side controls or
    CollegeVine controls. These are not reliable for inference and typically require manual review.

02_adequate_controls_base.csv
    The inverse of (01): rows where either crawl-side or CollegeVine controls were adequate.
    All remaining files (03-07) are subsets of this file.

03_strict_match.csv
    Adequate-controls subset where match_2013__match_level is strict_raw or strict_canonical.

04_non_strict_match.csv
    Adequate-controls subset where there is a match but it is not strict (e.g., fuzzy tiers,
    rescue tiers, related tiers).

05_no_match__2013_empty__now_something.csv
    Adequate-controls subset where there is NO_MATCH, the 2013 field is empty, and something
    exists in current discovered titles.

06_no_match__2013_something__now_nothing.csv
    Adequate-controls subset where there is NO_MATCH, something exists in 2013, and no current
    discovered titles were found.

07_no_match__2013_something__now_something_else.csv
    Adequate-controls subset where there is NO_MATCH, something exists in 2013, and some current
    discovered titles exist, but none matched.

Counting and Validation
----------------------
The script prints counts for each file and asserts:
  - (01 + 02) equals total rows
  - (03-07) are mutually exclusive and collectively exhaustive within (02)

Column Philosophy
-----------------
These simplified outputs emphasize:
  - 2013 program title
  - current matched title(s) and match metadata
  - high-level domain bucket hits
  - where new titles came from (crawl vs CollegeVine)
  - minimal control adequacy summary
Everything else is kept in the master CSV.
"""

# ============================================================
# PATCH: COLUMN ORDER + BUCKET SUMMARY NEAR 2013 PROGRAM NAME
# ============================================================

import os
import pandas as pd
import numpy as np

INPUT_CSV = "ace_unitid_merge__ace_x_2013comp__webscrape__v15simple__bucketed_programs__2013_current_matches.csv"
OUT_DIR = "simplified_splits"
os.makedirs(OUT_DIR, exist_ok=True)

ADEQUATE_CRAWL_SUFFICIENCY = {"sufficient majors", "ok", "passed_ctrl_thresh"}
ADEQUATE_CV_STATUS = {"passed_ctrl_thresh", "ok"}
TOTAL_CTRL_MIN = None  # e.g., 10

df = pd.read_csv(INPUT_CSV, dtype=str).fillna("")

if "uniqid" in df.columns and df["uniqid"].astype(str).str.strip().ne("").any():
    df["_uniqid"] = df["uniqid"].astype(str)
else:
    df["_uniqid"] = df["unitid"].astype(str) + "||" + df["2013_program_name"].astype(str)

def _norm(s: pd.Series) -> pd.Series:
    return s.astype(str).str.strip()

def _has_text(x: pd.Series) -> pd.Series:
    return x.astype(str).str.strip().ne("")

for c in [
    "controls_sufficiency",
    "college_vine_ctrl_status",
    "total_controls_found",
    "any_control_found",
    "match_2013__match_level",
    "match_2013__best_source",
    "2013_program_name",
    "discovered__all_titles",
]:
    if c in df.columns:
        df[c] = _norm(df[c])
    else:
        df[c] = ""

# ----------------------------
# Control adequacy split
# ----------------------------
crawl_ok = df["controls_sufficiency"].str.lower().isin({x.lower() for x in ADEQUATE_CRAWL_SUFFICIENCY})
cv_ok = df["college_vine_ctrl_status"].str.lower().isin({x.lower() for x in ADEQUATE_CV_STATUS})

if TOTAL_CTRL_MIN is not None:
    total_ctrl_num = pd.to_numeric(df["total_controls_found"], errors="coerce").fillna(0)
    crawl_ok = crawl_ok | (total_ctrl_num >= TOTAL_CTRL_MIN)

adequate_controls = crawl_ok | cv_ok
df_inadequate = df[~adequate_controls].copy()
df_adequate = df[adequate_controls].copy()

# ----------------------------
# Match splits (within adequate)
# ----------------------------
STRICT_LEVELS = {"strict_raw", "strict_canonical"}
is_strict = df_adequate["match_2013__match_level"].isin(STRICT_LEVELS)
is_no_match = df_adequate["match_2013__match_level"].eq("NO_MATCH")
is_some_match = (~is_no_match) & df_adequate["match_2013__match_level"].ne("")

now_has_any = _has_text(df_adequate["discovered__all_titles"])
if "discovered__all_titles" not in df_adequate.columns or df_adequate["discovered__all_titles"].eq("").all():
    parts = []
    for col in ["discovered__program_titles__crawl", "discovered__program_titles__cv", "discovered__signal_titles"]:
        if col in df_adequate.columns:
            parts.append(_has_text(df_adequate[col]))
    now_has_any = np.logical_or.reduce(parts) if parts else pd.Series(False, index=df_adequate.index)

has_2013 = _has_text(df_adequate["2013_program_name"])

df_strict = df_adequate[is_strict].copy()
df_non_strict_match = df_adequate[(~is_strict) & is_some_match].copy()
df_no_match_2013_empty_now_something = df_adequate[is_no_match & (~has_2013) & (now_has_any)].copy()
df_no_match_2013_something_now_nothing = df_adequate[is_no_match & (has_2013) & (~now_has_any)].copy()
df_no_match_2013_something_now_something_else = df_adequate[is_no_match & (has_2013) & (now_has_any)].copy()

# Sanity: partition df_adequate exactly
parts = [
    df_strict,
    df_non_strict_match,
    df_no_match_2013_empty_now_something,
    df_no_match_2013_something_now_nothing,
    df_no_match_2013_something_now_something_else,
]
part_ids = [set(x["_uniqid"].tolist()) for x in parts]
union_ids = set().union(*part_ids)
overlap = sum(len(part_ids[i].intersection(part_ids[j])) for i in range(len(part_ids)) for j in range(i + 1, len(part_ids)))

# ============================================================
# PATCH: SIMPLIFIED COLUMN SET + COLUMN ORDER
#   - Put "new vs old" columns immediately after 2013_program_name
#   - Include a bucket summary (program_buckets_hit preferred; fallback to program_bucket__*)
# ============================================================

# Bucket summary column: prefer program_buckets_hit, otherwise build from bucket flags if present
if "program_buckets_hit" not in df.columns:
    bucket_cols = [c for c in df.columns if c.startswith("program_bucket__") and not c.endswith(("__crawl", "__cv"))]
    if bucket_cols:
        def _bucket_summary_row(r):
            hits = []
            for c in bucket_cols:
                v = str(r.get(c, "")).strip()
                if v and v not in {"0", "False", "false", "nan", "None"}:
                    hits.append(c.replace("program_bucket__", ""))
            return "|".join(sorted(set(hits)))
        df["program_buckets_hit"] = df.apply(_bucket_summary_row, axis=1)
    else:
        df["program_buckets_hit"] = ""

# Define a reduced set, then enforce ordering.
KEEP_COLS = [
    # identity/context
    "unitid", "name", "2013_program_name",

    # NEW-vs-OLD summary block (right after 2013_program_name)
    "match_2013__best_title",          # "new name" (best match)
    "match_2013__best_source",         # crawl/cv/both/signal
    "match_2013__best_kind",           # program/signal
    "match_2013__match_level",         # strict/fuzzy/etc
    "match_2013__match_score",         # numeric score
    "program_buckets_hit",             # bucket summary
    "match_2013__detail",              # readable pair detail

    # current discovered titles (kept but moved later to avoid clutter)
    "discovered__program_titles__crawl",
    "discovered__program_titles__cv",
    "discovered__signal_titles",
    "discovered__all_titles",
    "discovered__new_titles_unmatched",
    "discovered__new_program_titles_when_best_signal",

    # minimal control summary
    "total_controls_found",
    "controls_sufficiency",
    "college_vine_ctrl_status",

    # hub + provenance
    "best_guess_inventory_url",
    "best_guess_inventory_reason",
    "college_vine_site",
    "college_vine_url",

    # optional location (kept, but not essential; place later)
    "Web_address", "city", "state", "control",
]

# Keep only columns that exist
KEEP_COLS = [c for c in KEEP_COLS if c in df.columns]

def simplify(d: pd.DataFrame) -> pd.DataFrame:
    out = d.loc[:, KEEP_COLS].copy()
    # enforce exact ordering in KEEP_COLS, then append any extras if you decide to keep them later
    return out

def write_csv(d: pd.DataFrame, filename: str):
    path = os.path.join(OUT_DIR, filename)
    simplify(d).to_csv(path, index=False)
    return path

paths = {}
paths["1_inadequate_controls"] = write_csv(df_inadequate, "01_inadequate_controls.csv")
paths["2_adequate_controls_base"] = write_csv(df_adequate, "02_adequate_controls_base.csv")
paths["3_strict_match"] = write_csv(df_strict, "03_strict_match.csv")
paths["4_non_strict_match"] = write_csv(df_non_strict_match, "04_non_strict_match.csv")
paths["5_no_match_2013_empty_now_something"] = write_csv(
    df_no_match_2013_empty_now_something,
    "05_no_match__2013_empty__now_something.csv",
)
paths["6_no_match_2013_something_now_nothing"] = write_csv(
    df_no_match_2013_something_now_nothing,
    "06_no_match__2013_something__now_nothing.csv",
)
paths["7_no_match_2013_something_now_something_else"] = write_csv(
    df_no_match_2013_something_now_something_else,
    "07_no_match__2013_something__now_something_else.csv",
)

# Counts + consistency checks
n_total = len(df)
n_inadequate = len(df_inadequate)
n_adequate = len(df_adequate)

counts = {
    "TOTAL_ROWS": n_total,
    "1_inadequate_controls": n_inadequate,
    "2_adequate_controls_base": n_adequate,
    "3_strict_match": len(df_strict),
    "4_non_strict_match": len(df_non_strict_match),
    "5_no_match__2013_empty__now_something": len(df_no_match_2013_empty_now_something),
    "6_no_match__2013_something__now_nothing": len(df_no_match_2013_something_now_nothing),
    "7_no_match__2013_something__now_something_else": len(df_no_match_2013_something_now_something_else),
    "ADEQUATE_PARTITION_SUM_(3-7)": sum(len(x) for x in parts),
    "ADEQUATE_UNION_UNIQIDS_(3-7)": len(union_ids),
    "ADEQUATE_OVERLAP_PAIRS_(should_be_0)": overlap,
}

print("=== OUTPUT FILES ===")
for k, v in paths.items():
    print(f"{k}: {v}")

print("\n=== COUNTS ===")
for k, v in counts.items():
    print(f"{k}: {v}")

assert n_inadequate + n_adequate == n_total, "Control split (01)+(02) does not sum to total."
assert overlap == 0, "One or more rows appear in multiple match buckets (03-07)."
assert len(union_ids) == n_adequate, "Buckets (03-07) do not cover all adequate-control rows."
assert sum(len(x) for x in parts) == n_adequate, "Bucket sizes (03-07) do not sum to adequate-control total."

print("\nAll consistency checks passed.")

=== OUTPUT FILES ===
1_inadequate_controls: simplified_splits/01_inadequate_controls.csv
2_adequate_controls_base: simplified_splits/02_adequate_controls_base.csv
3_strict_match: simplified_splits/03_strict_match.csv
4_non_strict_match: simplified_splits/04_non_strict_match.csv
5_no_match_2013_empty_now_something: simplified_splits/05_no_match__2013_empty__now_something.csv
6_no_match_2013_something_now_nothing: simplified_splits/06_no_match__2013_something__now_nothing.csv
7_no_match_2013_something_now_something_else: simplified_splits/07_no_match__2013_something__now_something_else.csv

=== COUNTS ===
TOTAL_ROWS: 357
1_inadequate_controls: 41
2_adequate_controls_base: 316
3_strict_match: 0
4_non_strict_match: 267
5_no_match__2013_empty__now_something: 0
6_no_match__2013_something__now_nothing: 39
7_no_match__2013_something__now_something_else: 10
ADEQUATE_PARTITION_SUM_(3-7): 316
ADEQUATE_UNION_UNIQIDS_(3-7): 316
ADEQUATE_OVERLAP_PAIRS_(should_be_0): 0

All consistency checks passed.

In [ ]:
# mermaid flow:
# flowchart TD
  A[Master CSV 2013 vs Current Matches] --> B{Adequate controls?}

  B -- No --> F1[01_inadequate_controls.csv
  No adequate controls in crawl or CV
  Manual review needed]

  B -- Yes --> C[02_adequate_controls_base.csv
  Base set for all subsequent splits]

  C --> D{Match level?}

  D -- Strict
  (strict_raw OR strict_canonical) --> F3[03_strict_match.csv]

  D -- Not strict but matched
  (match_level != NO_MATCH AND not strict) --> F4[04_non_strict_match.csv]

  D -- NO_MATCH --> E{2013 program present?}

  E -- 2013 empty --> E1{Any current titles found?}
  E1 -- Yes --> F5[05_no_match__2013_empty__now_something.csv
  New discovery with no 2013 baseline]
  E1 -- No --> FX1[Optional bucket
  NO_MATCH + 2013 empty + now empty]

  E -- 2013 present --> E2{Any current titles found?}
  E2 -- No --> F6[06_no_match__2013_something__now_nothing.csv
  2013 listed, nothing discovered now]
  E2 -- Yes --> F7[07_no_match__2013_something__now_something_else.csv
  Both exist but did not match]

IndentationError: unexpected indent (1458294461.py, line 2)